# Tire System Calculator

## Goal

Build and validate a Python calculator that ranks road tire pairings using
pressure, rolling resistance, mounted width, road surface and aerodynamic
performance. The notebook also packages the model as a static GitHub Pages app
for GitHub Pages.

## Setup

The notebook is the canonical source for the calculation model and the static
app. The final build step packages the same Python source without a separate
script.

In [1]:
import base64
import hashlib
import html
import io
import json
import math
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

In [2]:
gradio_runtime_imports = """import html
import json
import math
from pathlib import Path

import gradio as gr
import pandas as pd"""

## Steps

### 1. Define tire test inputs

In [3]:
tire_specs = pd.DataFrame(
    [
        {
            "tire_id": "str25",
            "name": "Continental GP5000 S TR 25",
            "short_name": "S TR 25",
            "family": "str",
            "nominal_width_mm": 25,
            "base_watts": 10.1,
            "reference_pressure_psi": 80,
            "aero_watts_at_40_kmh": -0.6,
            "width_model": "linear",
            "measured_width_mm": 25.3,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "str28",
            "name": "Continental GP5000 S TR 28",
            "short_name": "S TR 28",
            "family": "str",
            "nominal_width_mm": 28,
            "base_watts": 9.7,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": 0.0,
            "width_model": "linear",
            "measured_width_mm": 29.773,
            "reference_internal_width_mm": 23.5,
            "width_slope_per_mm": 0.3874,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "str30",
            "name": "Continental GP5000 S TR 30",
            "short_name": "S TR 30",
            "family": "str",
            "nominal_width_mm": 30,
            "base_watts": 10.0,
            "reference_pressure_psi": 69,
            "aero_watts_at_40_kmh": 1.1,
            "width_model": "linear",
            "measured_width_mm": 31.414,
            "reference_internal_width_mm": 23.5,
            "width_slope_per_mm": 0.3901,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "str32",
            "name": "Continental GP5000 S TR 32",
            "short_name": "S TR 32",
            "family": "str",
            "nominal_width_mm": 32,
            "base_watts": 9.8,
            "reference_pressure_psi": 64,
            "aero_watts_at_40_kmh": 2.55,
            "width_model": "linear",
            "measured_width_mm": 31.4,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "str35",
            "name": "Continental GP5000 S TR 35",
            "short_name": "S TR 35",
            "family": "str",
            "nominal_width_mm": 35,
            "base_watts": 9.6,
            "reference_pressure_psi": 58,
            "aero_watts_at_40_kmh": 6.45,
            "width_model": "linear",
            "measured_width_mm": 34.0,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "aero26",
            "name": "Continental AERO 111 26",
            "short_name": "AERO 111 26",
            "family": "aero111",
            "nominal_width_mm": 26,
            "base_watts": 10.5,
            "reference_pressure_psi": 80,
            "aero_watts_at_40_kmh": -0.29,
            "width_model": "linear",
            "measured_width_mm": 25.68,
            "reference_internal_width_mm": 22.0,
            "front_only": True,
            "confidence": "high",
        },
        {
            "tire_id": "aero29",
            "name": "Continental AERO 111 29",
            "short_name": "AERO 111 29",
            "family": "aero111",
            "nominal_width_mm": 29,
            "base_watts": 10.5,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": -1.23,
            "width_model": "linear",
            "measured_width_mm": 28.81,
            "reference_internal_width_mm": 22.0,
            "front_only": True,
            "confidence": "high",
        },
        {
            "tire_id": "slr28",
            "name": "Pirelli P Zero Race TLR SL-R 28",
            "short_name": "SL-R 28",
            "family": "slr",
            "nominal_width_mm": 28,
            "base_watts": 8.4,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": -1.05,
            "width_model": "slr28_box",
            "measured_width_mm": 28.5,
            "reference_internal_width_mm": 19.0,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "slr30",
            "name": "Pirelli P Zero Race TLR SL-R 30",
            "short_name": "SL-R 30",
            "family": "slr",
            "nominal_width_mm": 30,
            "base_watts": 8.5,
            "reference_pressure_psi": 67,
            "aero_watts_at_40_kmh": -0.6,
            "width_model": "linear",
            "measured_width_mm": 31.3,
            "reference_internal_width_mm": 23.5,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "tt28",
            "name": "Continental GP5000 TT TR 28",
            "short_name": "TT TR 28",
            "family": "tt",
            "nominal_width_mm": 28,
            "base_watts": 8.3,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": -0.3,
            "width_model": "linear",
            "measured_width_mm": 28.0,
            "reference_internal_width_mm": 19.0,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "tt30",
            "name": "Continental GP5000 TT TR 30",
            "short_name": "TT TR 30",
            "family": "tt",
            "nominal_width_mm": 30,
            "base_watts": 8.5,
            "reference_pressure_psi": 67,
            "aero_watts_at_40_kmh": -0.1,
            "width_model": "linear",
            "measured_width_mm": 31.3,
            "reference_internal_width_mm": 23.5,
            "front_only": False,
            "confidence": "high",
        },
    ]
).assign(
    width_slope_per_mm=lambda dataframe: dataframe["width_slope_per_mm"].fillna(
        0.4
    )
)

In [4]:
tire_specs.head()

,tire_id,name,short_name,family,nominal_width_mm,base_watts,reference_pressure_psi,aero_watts_at_40_kmh,width_model,measured_width_mm,reference_internal_width_mm,front_only,confidence,width_slope_per_mm
0,str25,Continental GP5000 S TR 25,S TR 25,str,25,10.1,80,-0.60,linear,25.300,17.8,False,high,0.4000
1,str28,Continental GP5000 S TR 28,S TR 28,str,28,9.7,72,0.00,linear,29.773,23.5,False,high,0.3874
2,str30,Continental GP5000 S TR 30,S TR 30,str,30,10.0,69,1.10,linear,31.414,23.5,False,high,0.3901
3,str32,Continental GP5000 S TR 32,S TR 32,str,32,9.8,64,2.55,linear,31.400,17.8,False,medium,0.4000
4,str35,Continental GP5000 S TR 35,S TR 35,str,35,9.6,58,6.45,linear,34.000,17.8,False,medium,0.4000


### 2. Define model assumptions

In [5]:
gravity = 9.80665
air_density_kg_m3 = 1.225
drivetrain_efficiency = 0.97
brr_load_kg = 42.5
brr_speed_mps = 29 / 3.6

surface_specs = {
    "New pavement": {"k1": 261.0, "roughness": 0.006},
    "Worn pavement": {"k1": 246.5, "roughness": 0.022},
    "Poor pavement": {"k1": 225.0, "roughness": 0.07},
    "Firm gravel": {"k1": 215.0, "roughness": 0.13},
    "Rough gravel": {"k1": 185.0, "roughness": 0.24},
    "Cobbles": {"k1": 199.0, "roughness": 0.5},
}

route_iri_reference = [
    (40.0, 261.0, 0.006),
    (80.0, 246.5, 0.022),
    (160.0, 225.0, 0.07),
    (250.0, 205.0, 0.16),
]

bike_specs = {
    "TT / triathlon": {
        "default_cda_m2": 0.23,
        "front_load_fraction": 0.5,
        "front_pressure_coefficient": 1.0,
        "rear_pressure_coefficient": 1.0,
    },
    "Road race": {
        "default_cda_m2": 0.30,
        "front_load_fraction": 0.45,
        "front_pressure_coefficient": 0.985,
        "rear_pressure_coefficient": 1.01,
    },
    "Endurance": {
        "default_cda_m2": 0.34,
        "front_load_fraction": 0.44,
        "front_pressure_coefficient": 0.975,
        "rear_pressure_coefficient": 1.02,
    },
    "Gravel race": {
        "default_cda_m2": 0.35,
        "front_load_fraction": 0.44,
        "front_pressure_coefficient": 0.96,
        "rear_pressure_coefficient": 1.02,
    },
}

kit_cda_adjustments_m2 = {
    "Fast trisuit": -0.015,
    "Standard trisuit": -0.005,
    "Jersey and bibs": 0.010,
}

### 3. Calculate pressure, width and power

In [6]:
def clamp(value, lower_bound, upper_bound):
    return min(upper_bound, max(lower_bound, value))


def estimated_cda_m2(
    bike_name,
    rider_height_cm,
    shoulder_width_cm,
    cockpit_width_cm,
    kit_type,
):
    body_scale = math.sqrt(
        rider_height_cm / 175 * shoulder_width_cm / 42
    )
    baseline_cda_m2 = bike_specs[bike_name][
        "default_cda_m2"
    ]
    cockpit_adjustment_m2 = (cockpit_width_cm - 40) * 0.0015

    return clamp(
        baseline_cda_m2 * body_scale
        + kit_cda_adjustments_m2[kit_type]
        + cockpit_adjustment_m2,
        0.16,
        0.50,
    )


def road_surface_profile(
    surface_name,
    route_iri_inches_per_mile=None,
):
    if route_iri_inches_per_mile is None:
        return surface_specs[surface_name]

    clipped_iri = clamp(
        route_iri_inches_per_mile,
        route_iri_reference[0][0],
        route_iri_reference[-1][0],
    )

    for lower_point, upper_point in zip(
        route_iri_reference,
        route_iri_reference[1:],
    ):
        if clipped_iri <= upper_point[0]:
            fraction = (
                (clipped_iri - lower_point[0])
                / (upper_point[0] - lower_point[0])
            )
            return {
                "k1": lower_point[1]
                + fraction * (upper_point[1] - lower_point[1]),
                "roughness": lower_point[2]
                + fraction * (upper_point[2] - lower_point[2]),
            }

    return surface_specs[surface_name]


def mounted_width_mm(tire, internal_width_mm):
    if tire["width_model"] == "slr28_box":
        if internal_width_mm <= 21:
            return 28.5 + 0.25 * (internal_width_mm - 19)
        return 29.0 + 0.5 * (internal_width_mm - 21)

    width_slope_per_mm = tire.get("width_slope_per_mm", 0.4)

    return tire["measured_width_mm"] + width_slope_per_mm * (
        internal_width_mm - tire["reference_internal_width_mm"]
    )


def silca_pressure_psi(
    system_weight_kg,
    mounted_width,
    speed_kmh,
    surface_name,
    bike_name,
    axle,
    wheel_bead_diameter_mm=622,
    route_iri_inches_per_mile=None,
):
    pressure_factor = (
        0.5 * (system_weight_kg - 50)
        + road_surface_profile(
            surface_name,
            route_iri_inches_per_mile,
        )["k1"]
    )
    tire_radius_term = mounted_width + wheel_bead_diameter_mm / 2
    pressure_numerator = (
        -0.00006 * mounted_width**3
        + 0.0079 * mounted_width**2
        - 0.4102 * mounted_width
        + 12.725
    ) * -226.44
    pressure_denominator_term = (
        (-0.5 * 9.81)
        / (pressure_factor * (20 / mounted_width))
        + tire_radius_term
    )
    contact_patch_pressure = pressure_numerator / (
        pressure_denominator_term**2 - tire_radius_term**2
    )
    speed_mph = speed_kmh / 1.609344
    speed_coefficient = 0.97 + ((speed_mph - 10) / 23) * 0.06
    pressure_coefficient = bike_specs[bike_name][
        f"{axle}_pressure_coefficient"
    ]

    return clamp(
        contact_patch_pressure
        * speed_coefficient
        * pressure_coefficient,
        28,
        120,
    )


def brr_reference_watts(tire, pressure_psi):
    return tire["base_watts"] * (
        tire["reference_pressure_psi"] / pressure_psi
    ) ** 0.12


def rolling_watts(tire, pressure_psi, wheel_load_kg, speed_kmh):
    reference_crr = brr_reference_watts(
        tire,
        pressure_psi,
    ) / (
        brr_load_kg * gravity * brr_speed_mps
    )
    return (
        reference_crr
        * wheel_load_kg
        * gravity
        * (speed_kmh / 3.6)
    )


def surface_watts(
    mounted_width,
    wheel_load_kg,
    speed_kmh,
    surface_name,
    route_iri_inches_per_mile=None,
):
    return (
        road_surface_profile(
            surface_name,
            route_iri_inches_per_mile,
        )["roughness"]
        * wheel_load_kg
        * (speed_kmh / 40) ** 2.2
        * (30 / mounted_width) ** 3
    )


def aero_watts(
    tire,
    mounted_width,
    external_width_mm,
    rim_depth_mm,
    speed_kmh,
    axle,
    internal_width_mm,
    air_density_kg_m3=1.225,
):
    speed_scale = (speed_kmh / 40) ** 3
    depth_scale = clamp(rim_depth_mm / 60, 0.65, 1.15)
    tire_rim_overlap = max(0, mounted_width - external_width_mm)

    if tire["family"] == "aero111":
        fit_rate = 0.10
    elif (
        tire["family"] == "slr"
        and 22 <= internal_width_mm <= 25
    ):
        fit_rate = 0.08
    else:
        fit_rate = 0.15

    base_aero_watts = tire["aero_watts_at_40_kmh"] * (
        depth_scale
        if tire["aero_watts_at_40_kmh"] < 0
        else 1.0
    ) + tire_rim_overlap * fit_rate

    return (
        base_aero_watts
        * speed_scale
        * (air_density_kg_m3 / 1.225)
        * (1.0 if axle == "front" else 0.2)
    )


def rank_tires(
    system_weight_kg,
    bike_name,
    surface_name,
    speed_from_kmh,
    speed_to_kmh,
    front_internal_width_mm,
    front_external_width_mm,
    front_rim_depth_mm,
    rear_internal_width_mm,
    rear_external_width_mm,
    rear_rim_depth_mm,
):
    lower_speed_kmh = min(speed_from_kmh, speed_to_kmh)
    upper_speed_kmh = max(speed_from_kmh, speed_to_kmh)
    speed_samples_kmh = [
        lower_speed_kmh
        + (upper_speed_kmh - lower_speed_kmh)
        * sample_number
        / 4
        for sample_number in range(5)
    ]
    midpoint_speed_kmh = (
        lower_speed_kmh + upper_speed_kmh
    ) / 2
    front_load_kg = (
        system_weight_kg
        * bike_specs[bike_name]["front_load_fraction"]
    )
    rear_load_kg = system_weight_kg - front_load_kg
    tires = tire_specs.to_dict("records")
    rear_tires = [
        tire for tire in tires if not tire["front_only"]
    ]
    result_rows = []

    for front_tire in tires:
        for rear_tire in rear_tires:
            front_width_mm = mounted_width_mm(
                front_tire,
                front_internal_width_mm,
            )
            rear_width_mm = mounted_width_mm(
                rear_tire,
                rear_internal_width_mm,
            )
            front_pressure_psi = silca_pressure_psi(
                system_weight_kg,
                front_width_mm,
                midpoint_speed_kmh,
                surface_name,
                bike_name,
                "front",
            )
            rear_pressure_psi = silca_pressure_psi(
                system_weight_kg,
                rear_width_mm,
                midpoint_speed_kmh,
                surface_name,
                bike_name,
                "rear",
            )
            rolling_samples_watts = []
            surface_samples_watts = []
            aero_samples_watts = []

            for speed_kmh in speed_samples_kmh:
                sample_front_pressure_psi = silca_pressure_psi(
                    system_weight_kg,
                    front_width_mm,
                    speed_kmh,
                    surface_name,
                    bike_name,
                    "front",
                )
                sample_rear_pressure_psi = silca_pressure_psi(
                    system_weight_kg,
                    rear_width_mm,
                    speed_kmh,
                    surface_name,
                    bike_name,
                    "rear",
                )
                rolling_samples_watts.append(
                    rolling_watts(
                        front_tire,
                        sample_front_pressure_psi,
                        front_load_kg,
                        speed_kmh,
                    )
                    + rolling_watts(
                        rear_tire,
                        sample_rear_pressure_psi,
                        rear_load_kg,
                        speed_kmh,
                    )
                )
                surface_samples_watts.append(
                    surface_watts(
                        front_width_mm,
                        front_load_kg,
                        speed_kmh,
                        surface_name,
                    )
                    + surface_watts(
                        rear_width_mm,
                        rear_load_kg,
                        speed_kmh,
                        surface_name,
                    )
                )
                aero_samples_watts.append(
                    aero_watts(
                        front_tire,
                        front_width_mm,
                        front_external_width_mm,
                        front_rim_depth_mm,
                        speed_kmh,
                        "front",
                        front_internal_width_mm,
                    )
                    + aero_watts(
                        rear_tire,
                        rear_width_mm,
                        rear_external_width_mm,
                        rear_rim_depth_mm,
                        speed_kmh,
                        "rear",
                        rear_internal_width_mm,
                    )
                )

            rolling_loss_watts = sum(
                rolling_samples_watts
            ) / len(rolling_samples_watts)
            surface_loss_watts = sum(
                surface_samples_watts
            ) / len(surface_samples_watts)
            aero_loss_watts = sum(
                aero_samples_watts
            ) / len(aero_samples_watts)
            result_rows.append(
                {
                    "front_tire": front_tire["short_name"],
                    "rear_tire": rear_tire["short_name"],
                    "front_pressure_psi": front_pressure_psi,
                    "rear_pressure_psi": rear_pressure_psi,
                    "front_width_mm": front_width_mm,
                    "rear_width_mm": rear_width_mm,
                    "rolling_watts": rolling_loss_watts,
                    "surface_watts": surface_loss_watts,
                    "aero_watts": aero_loss_watts,
                    "total_watts": (
                        rolling_loss_watts
                        + surface_loss_watts
                        + aero_loss_watts
                    ),
                    "confidence": (
                        "high"
                        if (
                            front_tire["confidence"] == "high"
                            and rear_tire["confidence"] == "high"
                        )
                        else "medium"
                    ),
                }
            )

    ranked_tires = (
        pd.DataFrame(result_rows)
        .sort_values("total_watts")
        .reset_index(drop=True)
        .assign(
            rank=lambda dataframe: dataframe.index + 1,
            gap_watts=lambda dataframe: (
                dataframe["total_watts"]
                - dataframe["total_watts"].min()
            ),
        )
    )

    return ranked_tires

### 4. Solve speed from rider power

In [7]:
def weather_for_speed(
    weather_profile,
    speed_kmh,
):
    if weather_profile is None:
        return air_density_kg_m3, 0.0

    speed_mph = max(speed_kmh / 1.609344, 1.0)
    sample_densities = []
    sample_yaws = []

    for weather_sample in weather_profile["samples"]:
        arrival_hour = min(
            23,
            int(
                (
                    weather_profile["start_time_minutes"]
                    + 60
                    * weather_sample["distance_miles"]
                    / speed_mph
                )
                / 60
            ),
        )
        air_density = weather_sample[
            "air_density_kg_m3"
        ][arrival_hour]
        sample_densities.append(air_density)

        bearing_radians = math.radians(
            weather_sample["bearing_degrees"]
        )
        rider_speed_mps = speed_kmh / 3.6
        rider_east_mps = rider_speed_mps * math.sin(
            bearing_radians
        )
        rider_north_mps = rider_speed_mps * math.cos(
            bearing_radians
        )
        apparent_east_mps = rider_east_mps - weather_sample[
            "wind_east_mps"
        ][arrival_hour]
        apparent_north_mps = rider_north_mps - weather_sample[
            "wind_north_mps"
        ][arrival_hour]
        dot_product = (
            rider_east_mps * apparent_east_mps
            + rider_north_mps * apparent_north_mps
        )
        cross_product = (
            rider_east_mps * apparent_north_mps
            - rider_north_mps * apparent_east_mps
        )
        sample_yaws.append(
            math.degrees(math.atan2(abs(cross_product), dot_product))
        )

    return (
        sum(sample_densities) / len(sample_densities),
        sum(sample_yaws) / len(sample_yaws),
    )


def rider_bike_aero_watts(
    rider_bike_cda_m2,
    speed_kmh,
    air_density_kg_m3=air_density_kg_m3,
):
    return (
        0.5
        * air_density_kg_m3
        * rider_bike_cda_m2
        * (speed_kmh / 3.6) ** 3
    )


def tire_system_losses(
    front_tire,
    rear_tire,
    front_width_mm,
    rear_width_mm,
    front_load_kg,
    rear_load_kg,
    bike_name,
    rider_bike_cda_m2,
    surface_name,
    front_internal_width_mm,
    front_external_width_mm,
    front_rim_depth_mm,
    rear_internal_width_mm,
    rear_external_width_mm,
    rear_rim_depth_mm,
    wheel_bead_diameter_mm,
    weather_profile,
    air_density_kg_m3,
    route_iri_inches_per_mile,
    speed_kmh,
):
    front_pressure_psi = silca_pressure_psi(
        front_load_kg / bike_specs[bike_name]["front_load_fraction"],
        front_width_mm,
        speed_kmh,
        surface_name,
        bike_name,
        "front",
        wheel_bead_diameter_mm,
        route_iri_inches_per_mile,
    )
    rear_pressure_psi = silca_pressure_psi(
        rear_load_kg / (1 - bike_specs[bike_name]["front_load_fraction"]),
        rear_width_mm,
        speed_kmh,
        surface_name,
        bike_name,
        "rear",
        wheel_bead_diameter_mm,
        route_iri_inches_per_mile,
    )
    front_rolling_watts = rolling_watts(
        front_tire,
        front_pressure_psi,
        front_load_kg,
        speed_kmh,
    )
    rear_rolling_watts = rolling_watts(
        rear_tire,
        rear_pressure_psi,
        rear_load_kg,
        speed_kmh,
    )
    front_surface_watts = surface_watts(
        front_width_mm,
        front_load_kg,
        speed_kmh,
        surface_name,
        route_iri_inches_per_mile,
    )
    rear_surface_watts = surface_watts(
        rear_width_mm,
        rear_load_kg,
        speed_kmh,
        surface_name,
        route_iri_inches_per_mile,
    )
    front_aero_watts = aero_watts(
        front_tire,
        front_width_mm,
        front_external_width_mm,
        front_rim_depth_mm,
        speed_kmh,
        "front",
        front_internal_width_mm,
        air_density_kg_m3,
    )
    rear_aero_watts = aero_watts(
        rear_tire,
        rear_width_mm,
        rear_external_width_mm,
        rear_rim_depth_mm,
        speed_kmh,
        "rear",
        rear_internal_width_mm,
        air_density_kg_m3,
    )

    return {
        "front_pressure_psi": front_pressure_psi,
        "rear_pressure_psi": rear_pressure_psi,
        "front_rolling_watts": front_rolling_watts,
        "rear_rolling_watts": rear_rolling_watts,
        "front_surface_watts": front_surface_watts,
        "rear_surface_watts": rear_surface_watts,
        "front_aero_watts": front_aero_watts,
        "rear_aero_watts": rear_aero_watts,
        "rolling_watts": front_rolling_watts + rear_rolling_watts,
        "surface_watts": front_surface_watts + rear_surface_watts,
        "aero_watts": front_aero_watts + rear_aero_watts,
    }


def solve_speed_kmh(
    rider_power_watts,
    tire_system_inputs,
):
    available_road_power_watts = rider_power_watts * drivetrain_efficiency
    lower_speed_kmh = 5.0
    upper_speed_kmh = 80.0

    for _ in range(40):
        midpoint_speed_kmh = (lower_speed_kmh + upper_speed_kmh) / 2
        effective_air_density_kg_m3, _ = weather_for_speed(
            tire_system_inputs["weather_profile"],
            midpoint_speed_kmh,
        )
        tire_losses = tire_system_losses(
            **tire_system_inputs,
            air_density_kg_m3=effective_air_density_kg_m3,
            speed_kmh=midpoint_speed_kmh,
        )
        required_road_power_watts = (
            rider_bike_aero_watts(
                tire_system_inputs["rider_bike_cda_m2"],
                midpoint_speed_kmh,
                effective_air_density_kg_m3,
            )
            + tire_losses["rolling_watts"]
            + tire_losses["surface_watts"]
            + tire_losses["aero_watts"]
        )

        if required_road_power_watts < available_road_power_watts:
            lower_speed_kmh = midpoint_speed_kmh
        else:
            upper_speed_kmh = midpoint_speed_kmh

    return (lower_speed_kmh + upper_speed_kmh) / 2


def rank_tires(
    system_weight_kg,
    rider_power_watts,
    wheel_bead_diameter_mm,
    bike_name,
    surface_name,
    front_internal_width_mm,
    front_external_width_mm,
    front_rim_depth_mm,
    rear_internal_width_mm,
    rear_external_width_mm,
    rear_rim_depth_mm,
    rider_height_cm=175,
    shoulder_width_cm=42,
    cockpit_width_cm=40,
    kit_type="Standard trisuit",
    route_iri_inches_per_mile=None,
    weather_profile=None,
):
    rider_bike_cda_m2 = estimated_cda_m2(
        bike_name,
        rider_height_cm,
        shoulder_width_cm,
        cockpit_width_cm,
        kit_type,
    )
    front_load_kg = (
        system_weight_kg
        * bike_specs[bike_name]["front_load_fraction"]
    )
    rear_load_kg = system_weight_kg - front_load_kg
    tires = tire_specs.to_dict("records")
    rear_tires = [
        tire for tire in tires if not tire["front_only"]
    ]
    result_rows = []

    for front_tire in tires:
        for rear_tire in rear_tires:
            front_width_mm = mounted_width_mm(
                front_tire,
                front_internal_width_mm,
            )
            rear_width_mm = mounted_width_mm(
                rear_tire,
                rear_internal_width_mm,
            )
            tire_system_inputs = {
                "front_tire": front_tire,
                "rear_tire": rear_tire,
                "front_width_mm": front_width_mm,
                "rear_width_mm": rear_width_mm,
                "front_load_kg": front_load_kg,
                "rear_load_kg": rear_load_kg,
                "bike_name": bike_name,
                "rider_bike_cda_m2": rider_bike_cda_m2,
                "surface_name": surface_name,
                "front_internal_width_mm": front_internal_width_mm,
                "front_external_width_mm": front_external_width_mm,
                "front_rim_depth_mm": front_rim_depth_mm,
                "rear_internal_width_mm": rear_internal_width_mm,
                "rear_external_width_mm": rear_external_width_mm,
                "rear_rim_depth_mm": rear_rim_depth_mm,
                "wheel_bead_diameter_mm": wheel_bead_diameter_mm,
                "weather_profile": weather_profile,
                "route_iri_inches_per_mile": route_iri_inches_per_mile,
            }
            predicted_speed_kmh = solve_speed_kmh(
                rider_power_watts,
                tire_system_inputs,
            )
            effective_air_density_kg_m3, mean_apparent_yaw_degrees = (
                weather_for_speed(
                    weather_profile,
                    predicted_speed_kmh,
                )
            )
            tire_losses = tire_system_losses(
                **tire_system_inputs,
                air_density_kg_m3=effective_air_density_kg_m3,
                speed_kmh=predicted_speed_kmh,
            )
            result_rows.append(
                {
                    "front_tire": front_tire["short_name"],
                    "rear_tire": rear_tire["short_name"],
                    "front_pressure_psi": tire_losses["front_pressure_psi"],
                    "rear_pressure_psi": tire_losses["rear_pressure_psi"],
                    "front_width_mm": front_width_mm,
                    "rear_width_mm": rear_width_mm,
                    "predicted_speed_kmh": predicted_speed_kmh,
                    "predicted_speed_mph": predicted_speed_kmh / 1.609344,
                    "air_density_kg_m3": effective_air_density_kg_m3,
                    "mean_apparent_yaw_degrees": mean_apparent_yaw_degrees,
                    "rider_power_watts": rider_power_watts,
                    "drivetrain_efficiency": drivetrain_efficiency,
                    "front_brr_reference_watts": brr_reference_watts(
                        front_tire,
                        tire_losses["front_pressure_psi"],
                    ),
                    "rear_brr_reference_watts": brr_reference_watts(
                        rear_tire,
                        tire_losses["rear_pressure_psi"],
                    ),
                    "front_rolling_watts": tire_losses["front_rolling_watts"],
                    "rear_rolling_watts": tire_losses["rear_rolling_watts"],
                    "front_surface_watts": tire_losses["front_surface_watts"],
                    "rear_surface_watts": tire_losses["rear_surface_watts"],
                    "front_aero_watts": tire_losses["front_aero_watts"],
                    "rear_aero_watts": tire_losses["rear_aero_watts"],
                    "rolling_watts": tire_losses["rolling_watts"],
                    "surface_watts": tire_losses["surface_watts"],
                    "aero_watts": tire_losses["aero_watts"],
                    "rider_bike_cda_m2": rider_bike_cda_m2,
                    "tire_loss_watts": (
                        tire_losses["rolling_watts"]
                        + tire_losses["surface_watts"]
                        + tire_losses["aero_watts"]
                    ),
                    "confidence": (
                        "high"
                        if (
                            front_tire["confidence"] == "high"
                            and rear_tire["confidence"] == "high"
                        )
                        else "medium"
                    ),
                }
            )

    ranked_tires = (
        pd.DataFrame(result_rows)
        .sort_values("predicted_speed_kmh", ascending=False)
        .reset_index(drop=True)
        .assign(
            rank=lambda dataframe: dataframe.index + 1,
            gap_speed_mph=lambda dataframe: (
                dataframe["predicted_speed_mph"].max()
                - dataframe["predicted_speed_mph"]
            ),
        )
    )

    return ranked_tires

### 4. Check the default scenario

In [8]:
default_ranking = rank_tires(
    system_weight_kg=90,
    rider_power_watts=200,
    wheel_bead_diameter_mm=622,
    bike_name="TT / triathlon",
    surface_name="Worn pavement",
    front_internal_width_mm=22,
    front_external_width_mm=31.5,
    front_rim_depth_mm=60,
    rear_internal_width_mm=22,
    rear_external_width_mm=31.5,
    rear_rim_depth_mm=60,
)

In [9]:
default_ranking.head(10)

,front_tire,rear_tire,front_pressure_psi,rear_pressure_psi,front_width_mm,rear_width_mm,predicted_speed_kmh,predicted_speed_mph,air_density_kg_m3,mean_apparent_yaw_degrees,...,front_aero_watts,rear_aero_watts,rolling_watts,surface_watts,aero_watts,rider_bike_cda_m2,tire_loss_watts,confidence,rank,gap_speed_mph
0,SL-R 28,SL-R 28,73.457269,73.457269,29.5,29.5,38.578790,23.971749,1.225,0.0,...,-0.942009,-0.188402,23.606913,1.923082,-1.130411,0.225,24.399584,high,1,0.000000
1,SL-R 28,TT TR 28,73.457257,74.657041,29.5,29.2,38.578693,23.971688,1.225,0.0,...,-0.942002,-0.053829,23.443684,1.953013,-0.995831,0.225,24.400867,medium,2,0.000060
2,SL-R 28,SL-R 30,73.456409,68.953126,29.5,30.7,38.571526,23.967235,1.225,0.0,...,-0.941477,-0.107597,23.730519,1.813926,-1.049075,0.225,24.495371,medium,3,0.004514
3,SL-R 28,TT TR 30,73.455642,68.952406,29.5,30.7,38.565038,23.963204,1.225,0.0,...,-0.941002,-0.017924,23.726558,1.813255,-0.958926,0.225,24.580887,high,4,0.008545
4,SL-R 30,SL-R 28,68.950535,73.453649,30.7,29.5,38.548191,23.952736,1.225,0.0,...,-0.537011,-0.187954,23.716270,1.811513,-0.724965,0.225,24.802818,medium,5,0.019013
5,SL-R 30,TT TR 28,68.950526,74.653364,30.7,29.2,38.548112,23.952686,1.225,0.0,...,-0.537008,-0.053701,23.553181,1.841395,-0.590709,0.225,24.803867,medium,6,0.019063
6,SL-R 30,SL-R 30,68.949729,68.949729,30.7,30.7,38.540939,23.948229,1.225,0.0,...,-0.536708,-0.107342,23.839759,1.702593,-0.644050,0.225,24.898302,medium,7,0.023520
7,TT TR 28,SL-R 28,74.652369,73.452661,29.2,29.5,38.539839,23.947546,1.225,0.0,...,-0.268331,-0.187832,23.420250,1.948689,-0.456163,0.225,24.912775,medium,8,0.024203
8,TT TR 28,TT TR 28,74.652360,74.652360,29.2,29.2,38.539764,23.947499,1.225,0.0,...,-0.268330,-0.053666,23.257199,1.978556,-0.321995,0.225,24.913759,medium,9,0.024249
9,SL-R 30,TT TR 30,68.949011,68.949011,30.7,30.7,38.534470,23.944210,1.225,0.0,...,-0.536438,-0.017881,23.835788,1.701965,-0.554319,0.225,24.983433,medium,10,0.027539


### 5. Build the Gradio interface

In [10]:
app_css = '''
.gradio-container {
    max-width: 980px !important;
    margin: 0 auto !important;
}
.section-label {
    color: #66706d;
    font-size: 0.72rem;
    font-weight: 700;
    letter-spacing: 0.12em;
    text-transform: uppercase;
}
.winner-card {
    background: #111b22;
    color: #ffffff;
    padding: 1.4rem;
    margin: 0.5rem 0 1rem;
}
.winner-kicker {
    color: #b9dc35;
    font-size: 0.72rem;
    font-weight: 700;
    letter-spacing: 0.1em;
    text-transform: uppercase;
}
.winner-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1rem;
    margin-top: 1rem;
}
.winner-grid h3 {
    font-size: 1.05rem;
    margin: 0.25rem 0 0.5rem;
}
.winner-grid p {
    color: #d4dcda;
    margin: 0;
}
.winner-total {
    border-top: 1px solid #445158;
    margin-top: 1rem;
    padding-top: 1rem;
}
.winner-total strong {
    color: #b9dc35;
    font-size: 1.65rem;
}
.model-note {
    color: #66706d;
    font-size: 0.82rem;
    line-height: 1.5;
}
@media (max-width: 640px) {
    .winner-grid {
        grid-template-columns: 1fr;
    }
}
'''


def format_outputs(ranked_tires):
    winner = ranked_tires.iloc[0]
    winner_html = f'''
    <section class="winner-card">
        <div class="winner-kicker">Fastest modeled system</div>
        <div class="winner-grid">
            <div>
                <small>FRONT</small>
                <h3>{winner["front_tire"]}</h3>
                <p>
                    {winner["front_pressure_psi"]:.1f} psi ·
                    {winner["front_width_mm"]:.1f} mm mounted
                </p>
            </div>
            <div>
                <small>REAR</small>
                <h3>{winner["rear_tire"]}</h3>
                <p>
                    {winner["rear_pressure_psi"]:.1f} psi ·
                    {winner["rear_width_mm"]:.1f} mm mounted
                </p>
            </div>
        </div>
        <div class="winner-total">
            <strong>{winner["predicted_speed_mph"]:.1f} mph</strong>
            <span> modeled steady speed at {winner["rider_power_watts"]:.0f} W</span>
        </div>
    </section>
    '''
    ranking_table = (
        ranked_tires
        .head(10)
        .loc[
            :,
            [
                "rank",
                "front_tire",
                "rear_tire",
                "predicted_speed_mph",
                "gap_speed_mph",
                "front_pressure_psi",
                "rear_pressure_psi",
                "front_width_mm",
                "rear_width_mm",
                "tire_loss_watts",
                "confidence",
            ],
        ]
        .rename(
            columns={
                "rank": "Rank",
                "front_tire": "Front",
                "rear_tire": "Rear",
                "predicted_speed_mph": "Speed mph",
                "gap_speed_mph": "Gap mph",
                "front_pressure_psi": "Front psi",
                "rear_pressure_psi": "Rear psi",
                "front_width_mm": "Front mm",
                "rear_width_mm": "Rear mm",
                "tire_loss_watts": "Tire loss W",
                "confidence": "Confidence",
            }
        )
        .round(
            {
                "Speed mph": 2,
                "Gap mph": 2,
                "Front psi": 1,
                "Rear psi": 1,
                "Front mm": 1,
                "Rear mm": 1,
                "Tire loss W": 1,
            }
        )
    )

    return winner_html, ranking_table


default_winner_html, default_ranking_table = format_outputs(
    default_ranking
)

In [11]:
gradio_interface_source = r"""with gr.Blocks(
    title="Tire System Calculator",
    css=app_css,
) as demo:
    gr.Markdown(
        "### INPUTS",
        elem_classes=["section-label"],
    )

    with gr.Row():
        with gr.Column():
            system_weight = gr.Slider(
                minimum=55,
                maximum=150,
                value=90,
                step=1,
                label="Total system weight (kg)",
            )
            bike = gr.Dropdown(
                choices=list(bike_specs),
                value="TT / triathlon",
                label="Bike position",
            )
            surface = gr.Dropdown(
                choices=list(surface_specs),
                value="Worn pavement",
                label="Road surface",
            )
            with gr.Row():
                speed_from = gr.Slider(
                    minimum=20,
                    maximum=65,
                    value=34,
                    step=1,
                    label="Speed from (km/h)",
                )
                speed_to = gr.Slider(
                    minimum=20,
                    maximum=65,
                    value=44,
                    step=1,
                    label="Speed to (km/h)",
                )

        with gr.Column():
            gr.Markdown("**Front wheel**")
            with gr.Row():
                front_internal_width = gr.Number(
                    value=22,
                    label="Internal width (mm)",
                )
                front_external_width = gr.Number(
                    value=31.5,
                    label="External width (mm)",
                )
                front_rim_depth = gr.Number(
                    value=60,
                    label="Rim depth (mm)",
                )

            gr.Markdown("**Rear wheel**")
            with gr.Row():
                rear_internal_width = gr.Number(
                    value=22,
                    label="Internal width (mm)",
                )
                rear_external_width = gr.Number(
                    value=31.5,
                    label="External width (mm)",
                )
                rear_rim_depth = gr.Number(
                    value=60,
                    label="Rim depth (mm)",
                )

    calculate_button = gr.Button(
        "Calculate fastest system",
        variant="primary",
    )

    gr.Markdown(
        "### RECOMMENDATION",
        elem_classes=["section-label"],
    )
    winner_output = gr.HTML(value=default_winner_html)
    ranking_output = gr.Dataframe(
        value=default_ranking_table,
        interactive=False,
        label="Top 10 front and rear pairings",
    )
    gr.Markdown(
        '''
        <p class="model-note">
        Aero is a relative adjustment against a well-matched GP5000 S TR 28
        setup. Rim fit is a modest continuous estimate, not a 105% rule.
        GP5000 S TR 28 and 30 mounted widths use 32 measurements from the
        Cyclingnews wheel-tunnel test: 29.8 and 31.4 mm at 23.5 mm internal
        width. Common bike and rider drag is excluded because it does not
        affect the ranking. Puncture risk is intentionally excluded.
        </p>
        '''
    )

    calculation_inputs = [
        system_weight,
        bike,
        surface,
        speed_from,
        speed_to,
        front_internal_width,
        front_external_width,
        front_rim_depth,
        rear_internal_width,
        rear_external_width,
        rear_rim_depth,
    ]

    calculate_button.click(
        fn=lambda *input_values: format_outputs(
            rank_tires(*input_values)
        ),
        inputs=calculation_inputs,
        outputs=[winner_output, ranking_output],
    )"""

### 6. Package the static Pages app

In [12]:
Path("../docs/assets/fixed").mkdir(parents=True, exist_ok=True)

gradio_wheel_url = (
    "https://cdn.jsdelivr.net/npm/@gradio/lite@5.45.0/dist/assets/"
    "gradio-5.45.0-cp312-none-any.whl"
)
huggingface_wheel_url = (
    "https://files.pythonhosted.org/packages/33/d5/"
    "d9e9b75d8dc9cf125fff16fb0cd51d864a29e8b46b6880d8808940989405/"
    "huggingface_hub-0.33.5-py3-none-any.whl"
)

with urllib.request.urlopen(gradio_wheel_url) as wheel_response:
    gradio_wheel_bytes = wheel_response.read()

with zipfile.ZipFile(io.BytesIO(gradio_wheel_bytes)) as source_wheel:
    wheel_files = {
        file_info.filename: source_wheel.read(file_info.filename)
        for file_info in source_wheel.infolist()
        if not file_info.is_dir()
    }

metadata_name = next(
    file_name
    for file_name in wheel_files
    if file_name.endswith(".dist-info/METADATA")
)
record_name = next(
    file_name
    for file_name in wheel_files
    if file_name.endswith(".dist-info/RECORD")
)
metadata_text = wheel_files[metadata_name].decode("utf-8").replace(
    "Requires-Dist: huggingface-hub<1.0,>=0.33.5",
    f"Requires-Dist: huggingface-hub @ {huggingface_wheel_url}",
)
wheel_files[metadata_name] = metadata_text.encode("utf-8")

record_rows = []
for file_name, file_bytes in sorted(wheel_files.items()):
    if file_name == record_name:
        continue
    file_digest = base64.urlsafe_b64encode(
        hashlib.sha256(file_bytes).digest()
    ).decode("ascii").rstrip("=")
    record_rows.append(
        f"{file_name},sha256={file_digest},{len(file_bytes)}"
    )
record_rows.append(f"{record_name},,")
wheel_files[record_name] = (
    "\n".join(record_rows) + "\n"
).encode("utf-8")

with zipfile.ZipFile(
    "../docs/assets/fixed/gradio-5.45.0-cp312-none-any.whl",
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as fixed_wheel:
    for file_name, file_bytes in sorted(wheel_files.items()):
        fixed_wheel.writestr(file_name, file_bytes)

with open("0_build.ipynb", encoding="utf-8") as notebook_file:
    saved_notebook = json.load(notebook_file)

gradio_model_source = "\n\n".join(
    "".join(cell["source"])
    for cell in saved_notebook["cells"]
    if (
        cell["cell_type"] == "code"
        and "gradio-lite" in cell.get("metadata", {}).get("tags", [])
    )
)
gradio_lite_python = (
    gradio_runtime_imports
    + "\n\n"
    + gradio_model_source
    + "\n\n"
    + gradio_interface_source
    + "\n\ndemo.launch()\n"
)

with open("../docs/index.html", "w", encoding="utf-8") as output_file:
    output_file.write(
        f'''<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="Compare road tires using pressure, rolling resistance, mounted width, road surface and aerodynamic performance.">
    <title>Tire System Calculator</title>
    <script>
        const NativeWorker = window.Worker;
        window.Worker = class extends NativeWorker {{
            postMessage(message, transfer) {{
                if (message?.type === "init-env") {{
                    const fixedMessage = structuredClone(message);
                    fixedMessage.data.gradioWheelUrl = new URL(
                        "./assets/fixed/gradio-5.45.0-cp312-none-any.whl",
                        window.location.href,
                    ).href;
                    return super.postMessage(fixedMessage, transfer);
                }}
                return super.postMessage(message, transfer);
            }}
        }};
    </script>
    <script type="module" crossorigin src="https://cdn.jsdelivr.net/npm/@gradio/lite@5.45.0/dist/lite.js"></script>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@gradio/lite@5.45.0/dist/lite.css">
    <style>
        * {{ box-sizing: border-box; }}
        body {{
            margin: 0;
            background: #f4f6f2;
            color: #101515;
            font-family: system-ui, sans-serif;
        }}
        header {{
            background: #111b22;
            color: white;
            padding: 3rem max(1.25rem, calc((100vw - 980px) / 2));
        }}
        header small {{
            color: #b9dc35;
            font-weight: 700;
            letter-spacing: 0.12em;
        }}
        header h1 {{
            font-size: clamp(2.2rem, 6vw, 4.6rem);
            letter-spacing: -0.055em;
            line-height: 0.96;
            margin: 0.8rem 0 1rem;
            max-width: 800px;
        }}
        header p {{
            color: #c8d0cd;
            line-height: 1.6;
            margin: 0;
            max-width: 650px;
        }}
        .loading-note {{
            color: #66706d;
            font-size: 0.8rem;
            margin: 1rem auto;
            max-width: 980px;
            padding: 0 1rem;
        }}
    </style>
</head>
<body>
    <header>
        <small>PYTHON · GRADIO LITE · MODEL 01</small>
        <h1>Find the fastest tire for your system.</h1>
        <p>
            Pressure, rolling resistance, mounted width and aerodynamic
            behavior evaluated together.
        </p>
    </header>
    <p class="loading-note">
        The Python model runs in your browser. The first load typically takes
        10 seconds.
    </p>
    <gradio-lite theme="light">
        <gradio-file name="app.py" entrypoint>
{html.escape(gradio_lite_python)}
        </gradio-file>
    </gradio-lite>
</body>
</html>
'''
    )

### 7. Build the dependency-free Pyodide interface

GitHub Pages cannot run a Python server. This build keeps every model
calculation in Python while using a small browser bridge for the interface.
It requires only the Pyodide runtime and installs no Python packages.

In [13]:
with open("0_build.ipynb", encoding="utf-8") as notebook_file:
    saved_notebook = json.load(notebook_file)

assumptions_source = "".join(
    next(
        cell["source"]
        for cell in saved_notebook["cells"]
        if cell.get("id") == "a205fa95"
    )
)
calculation_source = "\n\n".join(
    "".join(cell["source"])
    for cell in saved_notebook["cells"]
    if cell.get("id") in {"51b4adbe", "5df003b1"}
)
pandas_ranking_source = '''    ranked_tires = (
        pd.DataFrame(result_rows)
        .sort_values("total_watts")
        .reset_index(drop=True)
        .assign(
            rank=lambda dataframe: dataframe.index + 1,
            gap_watts=lambda dataframe: (
                dataframe["total_watts"]
                - dataframe["total_watts"].min()
            ),
        )
    )

    return ranked_tires'''
power_pandas_ranking_source = '''    ranked_tires = (
        pd.DataFrame(result_rows)
        .sort_values("predicted_speed_kmh", ascending=False)
        .reset_index(drop=True)
        .assign(
            rank=lambda dataframe: dataframe.index + 1,
            gap_speed_mph=lambda dataframe: (
                dataframe["predicted_speed_mph"].max()
                - dataframe["predicted_speed_mph"]
            ),
        )
    )

    return ranked_tires'''
power_python_ranking_source = '''    ranked_tires = sorted(
        result_rows,
        key=lambda result_row: result_row["predicted_speed_kmh"],
        reverse=True,
    )
    fastest_speed_mph = ranked_tires[0]["predicted_speed_mph"]
    for rank, result_row in enumerate(ranked_tires, start=1):
        result_row["rank"] = rank
        result_row["gap_speed_mph"] = (
            fastest_speed_mph - result_row["predicted_speed_mph"]
        )

    return ranked_tires'''
python_ranking_source = '''    ranked_tires = sorted(
        result_rows,
        key=lambda result_row: result_row["total_watts"],
    )
    fastest_watts = ranked_tires[0]["total_watts"]
    for rank, result_row in enumerate(ranked_tires, start=1):
        result_row["rank"] = rank
        result_row["gap_watts"] = (
            result_row["total_watts"] - fastest_watts
        )

    return ranked_tires'''
browser_calculation_source = (
    calculation_source
    .replace(
        '    tires = tire_specs.to_dict("records")',
        "    tires = tire_specs",
    )
    .replace(pandas_ranking_source, python_ranking_source)
    .replace(power_pandas_ranking_source, power_python_ranking_source)
)
assert "pd." not in browser_calculation_source

tire_specs_json = json.dumps(
    tire_specs.to_dict("records"),
    separators=(",", ":"),
)
browser_model_source = (
    "import json\nimport math\n\n"
    + f"tire_specs = json.loads({tire_specs_json!r})\n\n"
    + assumptions_source
    + "\n\n"
    + browser_calculation_source
    + '''


def calculate_json(input_json):
    calculation_inputs = json.loads(input_json)
    ranked_tires = rank_tires(**calculation_inputs)
    return json.dumps(ranked_tires)
'''
)

standalone_html = r'''<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="Compare road tires using pressure, rolling resistance, mounted width, road surface and aerodynamic performance.">
    <title>Tire System Calculator</title>
    <script src="https://cdn.jsdelivr.net/pyodide/v0.27.7/full/pyodide.js"></script>
    <style>
        :root {
            color-scheme: light;
            font-family: Inter, ui-sans-serif, system-ui, sans-serif;
        }
        * { box-sizing: border-box; }
        body {
            margin: 0;
            background: #f4f6f2;
            color: #101515;
        }
        header {
            background: #111b22;
            color: #ffffff;
            padding: 3rem max(1.25rem, calc((100vw - 1100px) / 2));
        }
        header small, .section-label {
            color: #9bbd22;
            font-size: 0.75rem;
            font-weight: 800;
            letter-spacing: 0.12em;
            text-transform: uppercase;
        }
        header h1 {
            font-size: clamp(2.3rem, 6vw, 4.8rem);
            letter-spacing: -0.055em;
            line-height: 0.96;
            margin: 0.8rem 0 1rem;
            max-width: 820px;
        }
        header p {
            color: #c8d0cd;
            line-height: 1.6;
            margin: 0;
            max-width: 680px;
        }
        main {
            margin: 0 auto;
            max-width: 1100px;
            padding: 1.5rem 1rem 4rem;
        }
        #runtime-status {
            color: #66706d;
            font-size: 0.88rem;
            margin: 0 0 1rem;
        }
        #runtime-status[data-state="error"] { color: #a71919; }
        .panel {
            background: #ffffff;
            border: 1px solid #dce2de;
            border-radius: 12px;
            padding: 1.25rem;
        }
        .wizard-nav { display: grid; gap: 0.6rem; grid-template-columns: repeat(4, minmax(0, 1fr)); margin-bottom: 1.25rem; }
        .wizard-tab { background: #f4f6f2; border-color: #dce2de; color: #4f5c58; margin: 0; min-height: 58px; text-align: left; }
        .wizard-tab[aria-current="step"] { background: #111b22; border-color: #111b22; color: #ffffff; }
        .wizard-step > h2 { margin-bottom: 0.35rem; }
        .wizard-intro { color: #66706d; font-size: 0.88rem; line-height: 1.5; margin: 0 0 1.25rem; max-width: 720px; }
        .wizard-actions { display: flex; gap: 0.75rem; justify-content: space-between; margin-top: 1.5rem; }
        .wizard-actions button { margin: 0; max-width: 260px; width: auto; }
        .wizard-actions .back-button { background: #ffffff; border-color: #111b22; color: #111b22; }
        .cda-preview { background: #e9efe3; border-radius: 8px; color: #35413e; font-size: 0.88rem; line-height: 1.5; margin: 1rem 0 0; padding: 0.85rem; }
        .inline-inputs { align-items: center; display: grid; gap: 0.45rem; grid-template-columns: minmax(0, 1fr) auto minmax(0, 1fr) auto; margin-top: 0.45rem; }
        .inline-inputs input { margin: 0; }
        .course-choice { background: #f4f6f2; border-radius: 8px; margin: 0 0 1rem; padding: 0.85rem; }
        .course-choice select { margin-top: 0.45rem; }
        .cda-preview strong { color: #111b22; }
        .input-grid {
            display: grid;
            gap: 1.5rem;
            grid-template-columns: repeat(2, minmax(0, 1fr));
        }
        .field-grid {
            display: grid;
            gap: 0.8rem;
            grid-template-columns: repeat(2, minmax(0, 1fr));
        }
        .wheel-grid {
            display: grid;
            gap: 0.8rem;
            grid-template-columns: repeat(3, minmax(0, 1fr));
        }
        h2, h3 { margin-top: 0; }
        h3 { margin-bottom: 0.75rem; }
        label {
            color: #35413e;
            display: grid;
            font-size: 0.8rem;
            font-weight: 700;
            gap: 0.35rem;
        }
        .field-help { color: #66706d; font-size: 0.72rem; font-weight: 500; line-height: 1.35; }
        input, select, button {
            border: 1px solid #bcc6c1;
            border-radius: 8px;
            font: inherit;
            min-height: 44px;
            padding: 0.65rem 0.75rem;
            width: 100%;
        }
        button {
            background: #111b22;
            border-color: #111b22;
            color: #ffffff;
            cursor: pointer;
            font-weight: 800;
            margin-top: 1rem;
        }
        button:disabled { cursor: wait; opacity: 0.55; }
        .results { margin-top: 1.5rem; }
        .winner-card {
            background: #111b22;
            border-radius: 12px;
            color: #ffffff;
            padding: 1.4rem;
        }
        .winner-grid {
            display: grid;
            gap: 1rem;
            grid-template-columns: repeat(2, minmax(0, 1fr));
            margin-top: 1rem;
        }
        .winner-card h3 { margin: 0.2rem 0 0.4rem; }
        .winner-card p { color: #d4dcda; margin: 0; }
        .winner-total {
            border-top: 1px solid #445158;
            margin-top: 1rem;
            padding-top: 1rem;
        }
        .winner-total strong { color: #b9dc35; font-size: 1.65rem; }
        .table-wrap {
            margin-top: 1rem;
            max-height: 560px;
            overflow: auto;
        }
        table {
            border-collapse: collapse;
            font-size: 0.8rem;
            width: 100%;
        }
        th, td {
            border-bottom: 1px solid #e1e6e3;
            padding: 0.65rem;
            text-align: right;
            white-space: nowrap;
        }
        th { background: #f4f6f2; position: sticky; top: 0; }
        th:nth-child(2), th:nth-child(3), td:nth-child(2), td:nth-child(3) {
            text-align: left;
        }
        .model-note {
            color: #66706d;
            font-size: 0.82rem;
            line-height: 1.5;
        }
        .route-panel {
            border-top: 1px solid #e1e6e3;
            margin-top: 1.5rem;
            padding-top: 1.25rem;
        }
        .route-panel button {
            background: #ffffff;
            border-color: #111b22;
            color: #111b22;
            margin-top: 0.8rem;
        }
        #route-status, #weather-status {
            color: #66706d;
            font-size: 0.82rem;
            line-height: 1.5;
            margin: 0.75rem 0 0;
        }
        #route-status[data-state="error"], #weather-status[data-state="error"] { color: #a71919; }
        #route-status[data-state="ready"], #weather-status[data-state="ready"] { color: #245f41; }
        .sources-panel { margin-top: 1.5rem; }
        .sources-panel summary { color: #111b22; cursor: pointer; font-size: 1.05rem; font-weight: 800; }
        .sources-panel details[open] summary { margin-bottom: 1rem; }
        .source-grid { display: grid; gap: 1.25rem; grid-template-columns: repeat(2, minmax(0, 1fr)); }
        .source-grid h3 { font-size: 0.95rem; margin-bottom: 0.5rem; }
        .source-grid p, .source-grid li { color: #4f5c58; font-size: 0.84rem; line-height: 1.5; }
        .source-grid ul { margin: 0; padding-left: 1.1rem; }
        .source-grid li + li { margin-top: 0.55rem; }
        .source-grid a { color: #245f41; font-weight: 700; }
        .source-grid .full-width { grid-column: 1 / -1; }
        .show-work-button { background: #ffffff; border-color: #111b22; color: #111b22; margin-top: 1rem; max-width: 230px; }
        #work-panel { margin-top: 1rem; }
        .work-recommendation { border-left: 4px solid #9bbd22; margin: 1rem 0; padding: 0.25rem 0 0.25rem 1rem; }
        .work-recommendation h3 { font-size: 1rem; margin-bottom: 0.45rem; }
        .work-recommendation p { color: #4f5c58; font-size: 0.89rem; line-height: 1.55; margin: 0.6rem 0; }
        .work-recommendation strong { color: #111b22; }
        .work-recommendation code { color: #111b22; display: block; font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 0.78rem; line-height: 1.55; overflow-wrap: anywhere; }
        .work-explainer { background: #e9efe3; border-radius: 8px; margin: 1rem 0; padding: 1rem; }
        .work-explainer h3 { font-size: 1rem; margin-bottom: 0.25rem; }
        .work-explainer > p { color: #4f5c58; font-size: 0.86rem; line-height: 1.5; margin: 0; }
        .work-steps { counter-reset: calculation-step; display: grid; gap: 0.7rem; grid-template-columns: repeat(3, minmax(0, 1fr)); list-style: none; margin: 1rem 0 0; padding: 0; }
        .work-steps li { background: #ffffff; border-radius: 8px; color: #4f5c58; counter-increment: calculation-step; font-size: 0.82rem; line-height: 1.45; padding: 0.8rem; }
        .work-steps li::before { color: #245f41; content: counter(calculation-step) ". "; font-weight: 800; }
        .work-steps strong { color: #111b22; }
        .work-grid { display: grid; gap: 1rem; grid-template-columns: repeat(2, minmax(0, 1fr)); }
        .work-card { background: #f4f6f2; border-radius: 8px; padding: 1rem; }
        .work-card h3 { font-size: 0.95rem; margin-bottom: 0.55rem; }
        .work-card p { color: #4f5c58; font-size: 0.83rem; line-height: 1.5; margin: 0.55rem 0 0; }
        .work-card code { color: #111b22; display: block; font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 0.78rem; line-height: 1.55; overflow-wrap: anywhere; }
        .work-values { display: grid; gap: 0.4rem; grid-template-columns: repeat(2, minmax(0, 1fr)); }
        .work-values span { color: #4f5c58; font-size: 0.8rem; }
        .work-values strong { color: #111b22; display: block; font-size: 0.92rem; }
        @media (max-width: 760px) {
            .input-grid, .winner-grid, .wizard-nav { grid-template-columns: 1fr; }
            .wheel-grid, .source-grid, .work-grid, .work-steps { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body data-ready="false">
    <header>
        <small>PYTHON · PYODIDE · MODEL 01</small>
        <h1>Find the fastest tire for your system.</h1>
        <p>Pressure, rolling resistance, mounted width and aerodynamic behavior evaluated together.</p>
    </header>
    <main>
        <p id="runtime-status" data-state="loading">Loading the Python model…</p>
        <section class="panel">
            <nav class="wizard-nav" aria-label="Calculator steps">
                <button class="wizard-tab" type="button" data-wizard-target="1" aria-current="step">1. Tire pressure</button>
                <button class="wizard-tab" type="button" data-wizard-target="2">2. Rider and bike aero</button>
                <button class="wizard-tab" type="button" data-wizard-target="3">3. Course and weather</button>
                <button id="results-tab" class="wizard-tab" type="button" data-wizard-target="4" disabled>4. Results</button>
            </nav>
            <section class="wizard-step" data-wizard-step="1">
                <p class="section-label">Step 1 of 4</p>
                <h2>Set up the tire-pressure model</h2>
                <p class="wizard-intro">System weight, wheel diameter and internal rim width determine the tire’s air volume and the pressure calculation. The calculator will test every compatible tire pairing after you finish the remaining steps.</p>
                <div class="input-grid">
                    <div class="field-grid">
                        <label>Total system weight (lb)<input id="system-weight" type="number" min="120" max="330" step="1" value="198"></label>
                        <label>Wheel size<select id="wheel-size"><option value="622">700c / 29 in</option><option value="584">650b / 27.5 in</option></select></label>
                    </div>
                    <div>
                        <h3>Internal rim width</h3>
                        <div class="field-grid">
                            <label>Front internal width (mm)<input id="front-internal" type="number" step="0.1" value="22"></label>
                            <label>Rear internal width (mm)<input id="rear-internal" type="number" step="0.1" value="22"></label>
                        </div>
                    </div>
                </div>
                <div class="wizard-actions"><span></span><button type="button" data-wizard-next="2">Next: rider and bike aero</button></div>
            </section>
            <section class="wizard-step" data-wizard-step="2" hidden>
                <p class="section-label">Step 2 of 4</p>
                <h2>Estimate rider and bike aero</h2>
                <p class="wizard-intro">These fields set a transparent CdA estimate. It is directional, not a wind-tunnel measurement: body size and kit alter frontal area while cockpit width influences your effective position.</p>
                <div class="input-grid">
                    <div class="field-grid">
                        <label>Average pedal power for the course (W)<input id="rider-power" type="number" min="100" max="500" step="5" value="200"><span class="field-help">Use expected course average, not Normalized Power (NP).</span></label>
                        <label>Bike position<select id="bike"><option>TT / triathlon</option><option>Road race</option><option>Endurance</option><option>Gravel race</option></select></label>
                        <label>Height<div class="inline-inputs"><input id="rider-height-feet" aria-label="Height (ft)" type="number" min="4" max="8" step="1" value="5"><span>ft</span><input id="rider-height-inches" aria-label="Height (in)" type="number" min="0" max="11" step="1" value="9"><span>in</span></div></label>
                        <label>Shoulder width (cm)<input id="shoulder-width" type="number" min="30" max="60" step="0.5" value="43"></label>
                        <label>Handlebar / arm-pad width (cm)<input id="cockpit-width" type="number" min="30" max="56" step="0.5" value="41"></label>
                        <label>Kit type<select id="kit-type"><option>Fast trisuit</option><option selected>Standard trisuit</option><option>Jersey and bibs</option></select></label>
                    </div>
                    <div>
                        <p class="section-label">Wheel aero geometry</p>
                        <h3>Front wheel</h3>
                        <div class="field-grid">
                            <label>External width (mm)<input id="front-external" type="number" step="0.1" value="31.5"></label>
                            <label>Rim depth (mm)<input id="front-depth" type="number" step="1" value="60"></label>
                        </div>
                        <h3 style="margin-top: 1rem">Rear wheel</h3>
                        <div class="field-grid">
                            <label>External width (mm)<input id="rear-external" type="number" step="0.1" value="31.5"></label>
                            <label>Rim depth (mm)<input id="rear-depth" type="number" step="1" value="60"></label>
                        </div>
                        <p id="cda-preview" class="cda-preview"></p>
                    </div>
                </div>
                <div class="wizard-actions"><button class="back-button" type="button" data-wizard-next="1">Back: tire pressure</button><button type="button" data-wizard-next="3">Next: course and weather</button></div>
            </section>
            <section class="wizard-step" data-wizard-step="3" hidden>
                <p class="section-label">Step 3 of 4</p>
                <h2>Describe the course</h2>
                <p class="wizard-intro">Choose pavement condition for a quick estimate, or upload a GPX to use route-matched road roughness and historical weather.</p>
                <div class="course-choice">
                    <label>Do you want to upload a GPX?<select id="course-mode"><option value="pavement">No — choose pavement condition</option><option value="gpx">Yes — upload a GPX</option></select></label>
                </div>
                <div id="pavement-panel">
                    <label>Pavement condition<select id="surface"><option>New pavement</option><option selected>Worn pavement</option><option>Poor pavement</option><option>Firm gravel</option><option>Rough gravel</option><option>Cobbles</option></select></label>
                </div>
                <div id="gpx-panel" hidden>
                    <div class="route-panel">
                        <p class="section-label">Course roughness</p>
                        <label>GPX course file<input id="route-file" type="file" accept=".gpx,application/gpx+xml,application/xml,text/xml"></label>
                        <p id="route-status" data-state="idle">Upload a GPX to match roughly one-mile route samples to FHWA HPMS road-roughness data.</p>
                        <button id="analyze-route-button" type="button" disabled>Match GPX to road roughness</button>
                    </div>
                    <div id="weather-panel" class="route-panel">
                        <p class="section-label">Historical weather and yaw (optional)</p>
                        <div class="field-grid">
                            <label>Race date<input id="race-date" type="date"></label>
                            <label>What time will you start biking?<input id="race-start-time" type="time" value="07:00"></label>
                        </div>
                        <p id="weather-status" data-state="idle">Choose a race date after uploading the GPX. The model uses the same calendar date in each of the three prior years.</p>
                        <button id="analyze-weather-button" type="button" disabled>Use historic weather for this date</button>
                    </div>
                </div>
                <div class="wizard-actions"><button class="back-button" type="button" data-wizard-next="2">Back: rider and bike aero</button><button id="calculate-button" type="button" disabled>Loading Python…</button></div>
            </section>
            <section id="results-section" class="wizard-step" data-wizard-step="4" aria-live="polite" hidden>
            <p class="section-label">Step 4 of 4</p>
            <h2>Your results</h2>
            <p class="section-label">Recommendation</p>
            <div id="winner-output"></div>
            <div class="table-wrap"><table id="ranking-table"></table></div>
            <p class="model-note">Speed is solved from average power at the pedals and a transparent CdA estimate based on riding position, height, shoulder width, cockpit width and kit type. Treat absolute mph as an estimate and compare the tire-to-tire speed gaps. A matched GPX uses FHWA HPMS International Roughness Index (IRI) measurements where they are available. Historic weather uses the selected date’s three prior calendar years. The power-solved speed determines each route sample’s arrival hour, which supplies air density to the speed model and produces a reported apparent-wind yaw. Published tests do show yaw behavior, but this catalog currently stores one aero baseline per tire rather than a digitized tire-and-rim curve at each yaw angle. IRI is translated into this model’s pressure and roughness curve, not treated as a direct laboratory rolling-resistance test. Gravel race and gravel surfaces use approximate position and roughness inputs, but the current catalog contains road tires through 35 mm rather than gravel-tire tests. Aero is a relative adjustment against a well-matched GP5000 S TR 28 setup. Rim fit is a modest continuous estimate, not a 105% rule. GP5000 S TR 28 and 30 mounted widths use 32 measurements from the Cyclingnews wheel-tunnel test: 29.8 and 31.4 mm at 23.5 mm internal width. Puncture risk is intentionally excluded.</p>
            <button id="show-work-button" class="show-work-button" type="button" disabled aria-expanded="false">Show your work</button>
            <section id="work-panel" class="panel" data-testid="work-panel" hidden></section>
            <div class="wizard-actions"><button class="back-button" type="button" data-wizard-next="3">Back: course and weather</button><span></span></div>
            </section>
        </section>

        <section class="panel sources-panel" data-testid="data-sources">
            <p class="section-label">Sources and model status</p>
            <details open>
                <summary>What each input is based on</summary>
                <div class="source-grid">
                    <div>
                        <h3>Published or measured inputs</h3>
                        <ul>
                            <li><a href="https://www.bicyclerollingresistance.com/" target="_blank" rel="noreferrer">Bicycle Rolling Resistance</a> supplies the rolling-resistance anchors. They are manually transcribed and normalized to this model’s 42.5 kg, 29 km/h reference, not fetched live.</li>
                            <li><a href="https://www.cyclingnews.com/cycling-tech-components/wheels-tyres/what-are-the-fastest-uci-legal-road-wheels-wind-tunnel-testing-the-big-name-brands-and-chinese-contenders/" target="_blank" rel="noreferrer">Cyclingnews wheel-tunnel test</a> supplies the GP5000 S TR 28 and 30 mounted-width calibration. The calibration notebook transcribes its 32 width measurements.</li>
                            <li>Pirelli SL-R 28 WAM uses the package chart supplied for this project: 28.5, 29.0 and 30.0 mm on 19c, 21c and 23c ETRTO rims. The SL-R 30 mounted-width input is from the <a href="https://www.parcours.cc/blogs/news/aero-testing-race-tyres-2026-update" target="_blank" rel="noreferrer">Parcours 2026 test</a>.</li>
                            <li><a href="https://data.transportation.gov/Roadways-and-Bridges/HPMS-Spatial-All-Sections-2024/42um-tgh5" target="_blank" rel="noreferrer">FHWA HPMS 2024</a> provides matched road IRI. <a href="https://open-meteo.com/en/docs/historical-weather-api" target="_blank" rel="noreferrer">Open-Meteo Archive</a> provides historical temperature, pressure and wind.</li>
                        </ul>
                    </div>
                    <div>
                        <h3>Derived model inputs</h3>
                        <ul>
                            <li>The <a href="https://silca.cc/pages/pro-tire-pressure-calculator" target="_blank" rel="noreferrer">SILCA calculator</a> informed the pressure relationship. This is a reconstructed approximation, not an official SILCA API or a claim of identical outputs.</li>
                            <li>CdA from bike position, rider height, shoulder width, cockpit width and kit type, the 97% drivetrain factor, surface-loss mapping, IRI-to-pressure mapping, width extrapolation outside measured rim widths and rim-fit penalties are calculator assumptions. They are shown as estimates, not external measurements.</li>
                            <li>The tire-rim aero model applies the equivalent front effect at 1.0 and rear effect at 0.2. That 5:1 exposure weighting is an explicit model assumption, not a direct universal wind-tunnel result.</li>
                            <li>Rolling inputs for tires without an equivalent published protocol are provisional comparisons. They should not be read as a fresh lab test of that tire.</li>
                        </ul>
                    </div>
                    <div class="full-width">
                        <h3>Yaw: what the calculator does and does not have</h3>
                        <p><a href="https://www.parcours.cc/blogs/news/aero-testing-race-tyres-2026-update" target="_blank" rel="noreferrer">Parcours’ 2026 wind-tunnel test</a> is yaw-resolved: it tested a 62 mm, 23.5 mm-internal front wheel at 72 psi and reports that AERO 111 and SL-R separate from conventional tires above 10° yaw. Its conventional tire group supports treating GP5000 TT TR and S TR alike for yaw behavior. <a href="https://www.cyclingnews.com/cycling-tech-components/wheels-tyres/aero-tyre-focus/" target="_blank" rel="noreferrer">Cyclingnews’ AERO 111 test</a> also measures seven yaw angles and provides the 40 km/h AERO 111 baselines used here.</p>
                        <p>The catalog itself is not yet yaw-resolved: it holds a single 40 km/h aero baseline per tire, not a numeric 0°, 5°, 10° and 15° curve for every tire on every rim. Therefore route weather changes air density and reports the physically calculated yaw, but the model does not invent a transferable tire-specific yaw multiplier. The current aero comparison uses the same conventional-yaw treatment for GP5000 TT TR and S TR; AERO 111 and SL-R are identified as the high-yaw exceptions in the source material. Nero Show discussion is not used as an input because no primary, reproducible tire-and-wheel wind-tunnel dataset has been identified.</p>
                    </div>
                </div>
            </details>
        </section>
    </main>
    <script>
        const pythonModelSource = __PYTHON_MODEL_SOURCE__;
        const runtimeStatus = document.getElementById("runtime-status");
        const calculateButton = document.getElementById("calculate-button");
        const routeFileInput = document.getElementById("route-file");
        const analyzeRouteButton = document.getElementById("analyze-route-button");
        const routeStatus = document.getElementById("route-status");
        const raceDateInput = document.getElementById("race-date");
        const raceStartTimeInput = document.getElementById("race-start-time");
        const analyzeWeatherButton = document.getElementById("analyze-weather-button");
        const weatherStatus = document.getElementById("weather-status");
        const cdaPreview = document.getElementById("cda-preview");
        const courseMode = document.getElementById("course-mode");
        const pavementPanel = document.getElementById("pavement-panel");
        const gpxPanel = document.getElementById("gpx-panel");
        const wizardTabs = Array.from(document.querySelectorAll("[data-wizard-target]"));
        const wizardSteps = Array.from(document.querySelectorAll("[data-wizard-step]"));
        const resultsTab = document.getElementById("results-tab");
        const showWorkButton = document.getElementById("show-work-button");
        const workPanel = document.getElementById("work-panel");
        const hpmsApiUrl = "https://data.transportation.gov/resource/42um-tgh5.json";
        const weatherApiUrl = "https://archive-api.open-meteo.com/v1/archive";
        const metersPerMile = 1609.344;
        let pythonRuntime;
        let routePoints = [];
        let routeProfile = null;
        let weatherProfile = null;
        let latestWinner = null;
        let latestInputs = null;

        function estimatedCdaPreviewM2() {
            const baselineCda = {
                "TT / triathlon": 0.23,
                "Road race": 0.30,
                Endurance: 0.34,
                "Gravel race": 0.35
            }[document.getElementById("bike").value];
            const kitAdjustment = {
                "Fast trisuit": -0.015,
                "Standard trisuit": -0.005,
                "Jersey and bibs": 0.010
            }[document.getElementById("kit-type").value];
            const heightCm = numericValue("rider-height-feet") * 30.48
                + numericValue("rider-height-inches") * 2.54;
            const shoulderWidthCm = numericValue("shoulder-width");
            const cockpitWidthCm = numericValue("cockpit-width");
            const bodyScale = Math.sqrt(heightCm / 175 * shoulderWidthCm / 42);

            return Math.min(0.50, Math.max(
                0.16,
                baselineCda * bodyScale + kitAdjustment
                    + (cockpitWidthCm - 40) * 0.0015
            ));
        }

        function updateCdaPreview() {
            cdaPreview.innerHTML = "Estimated CdA: <strong>"
                + estimatedCdaPreviewM2().toFixed(3)
                + " m²</strong>. This is a transparent directional estimate, not a personal aero test.";
        }

        function updateCourseMode() {
            const usingGpx = courseMode.value === "gpx";
            pavementPanel.hidden = usingGpx;
            gpxPanel.hidden = !usingGpx;
            if (!usingGpx) {
                routeFileInput.value = "";
                routePoints = [];
                routeProfile = null;
                weatherProfile = null;
                analyzeRouteButton.disabled = true;
                updateRouteStatus("Choose the GPX option to match national road-roughness data.");
                updateWeatherStatus("Choose the GPX option to use historic weather and yaw.");
            }
            updateWeatherButton();
        }

        function showWizardStep(stepNumber) {
            wizardSteps.forEach((step) => {
                step.hidden = Number(step.dataset.wizardStep) !== stepNumber;
            });
            wizardTabs.forEach((tab) => {
                const isCurrentStep = Number(tab.dataset.wizardTarget) === stepNumber;
                if (isCurrentStep) {
                    tab.setAttribute("aria-current", "step");
                } else {
                    tab.removeAttribute("aria-current");
                }
            });
            if (stepNumber === 2) {
                updateCdaPreview();
            }
        }

        wizardTabs.forEach((tab) => {
            tab.addEventListener("click", () => {
                showWizardStep(Number(tab.dataset.wizardTarget));
            });
        });

        document.querySelectorAll("[data-wizard-next]").forEach((button) => {
            button.addEventListener("click", () => {
                showWizardStep(Number(button.dataset.wizardNext));
            });
        });

        ["bike", "rider-height-feet", "rider-height-inches", "shoulder-width", "cockpit-width", "kit-type"].forEach((elementId) => {
            document.getElementById(elementId).addEventListener("input", updateCdaPreview);
            document.getElementById(elementId).addEventListener("change", updateCdaPreview);
        });

        courseMode.addEventListener("change", updateCourseMode);

        updateCdaPreview();
        updateCourseMode();

        function updateRouteStatus(message, state = "idle") {
            routeStatus.textContent = message;
            routeStatus.dataset.state = state;
        }

        function updateWeatherStatus(message, state = "idle") {
            weatherStatus.textContent = message;
            weatherStatus.dataset.state = state;
        }

        function updateWeatherButton() {
            analyzeWeatherButton.disabled = (
                !pythonRuntime
                || courseMode.value !== "gpx"
                || routePoints.length < 2
                || !raceDateInput.value
            );
        }

        function haversineMeters(firstPoint, secondPoint) {
            const radians = Math.PI / 180;
            const latitudeDelta = (secondPoint.latitude - firstPoint.latitude) * radians;
            const longitudeDelta = (secondPoint.longitude - firstPoint.longitude) * radians;
            const latitudeStart = firstPoint.latitude * radians;
            const latitudeEnd = secondPoint.latitude * radians;
            const haversine = Math.sin(latitudeDelta / 2) ** 2
                + Math.cos(latitudeStart) * Math.cos(latitudeEnd)
                * Math.sin(longitudeDelta / 2) ** 2;
            return 6371000 * 2 * Math.atan2(Math.sqrt(haversine), Math.sqrt(1 - haversine));
        }

        function routeBearingDegrees(firstPoint, secondPoint) {
            const radians = Math.PI / 180;
            const longitudeDelta = (secondPoint.longitude - firstPoint.longitude) * radians;
            const firstLatitude = firstPoint.latitude * radians;
            const secondLatitude = secondPoint.latitude * radians;
            const eastComponent = Math.sin(longitudeDelta) * Math.cos(secondLatitude);
            const northComponent = Math.cos(firstLatitude) * Math.sin(secondLatitude)
                - Math.sin(firstLatitude) * Math.cos(secondLatitude)
                * Math.cos(longitudeDelta);
            return (Math.atan2(eastComponent, northComponent) / radians + 360) % 360;
        }

        function parseGpxRoute(gpxText) {
            const gpxDocument = new DOMParser().parseFromString(gpxText, "application/xml");
            if (gpxDocument.querySelector("parsererror")) {
                throw new Error("The uploaded file is not valid GPX.");
            }
            const gpxPoints = Array.from(
                gpxDocument.querySelectorAll("trkpt, rtept")
            ).map((point) => ({
                latitude: Number(point.getAttribute("lat")),
                longitude: Number(point.getAttribute("lon"))
            })).filter((point) => (
                Number.isFinite(point.latitude)
                && Number.isFinite(point.longitude)
            ));
            if (gpxPoints.length < 2) {
                throw new Error("The GPX needs at least two route points.");
            }
            return gpxPoints;
        }

        function routeSamplesFromPoints(points) {
            const routeSegments = points.slice(1).map((point, index) => ({
                start: points[index],
                end: point,
                distanceMeters: haversineMeters(points[index], point)
            })).filter((segment) => segment.distanceMeters > 0);
            const routeDistanceMeters = routeSegments.reduce(
                (total, segment) => total + segment.distanceMeters,
                0
            );
            if (routeDistanceMeters <= 0) {
                throw new Error("The GPX route has no measurable distance.");
            }
            const sampleCount = Math.max(
                1,
                Math.ceil(routeDistanceMeters / metersPerMile)
            );
            const sampleSpacingMeters = routeDistanceMeters / sampleCount;
            const routeSamples = [];
            let segmentIndex = 0;
            let distanceBeforeSegment = 0;

            for (let sampleIndex = 0; sampleIndex < sampleCount; sampleIndex += 1) {
                const targetDistance = (sampleIndex + 0.5) * sampleSpacingMeters;
                while (
                    segmentIndex < routeSegments.length - 1
                    && distanceBeforeSegment + routeSegments[segmentIndex].distanceMeters < targetDistance
                ) {
                    distanceBeforeSegment += routeSegments[segmentIndex].distanceMeters;
                    segmentIndex += 1;
                }
                const segment = routeSegments[segmentIndex];
                const segmentFraction = Math.min(
                    1,
                    Math.max(
                        0,
                        (targetDistance - distanceBeforeSegment) / segment.distanceMeters
                    )
                );
                routeSamples.push({
                    latitude: segment.start.latitude
                        + segmentFraction * (segment.end.latitude - segment.start.latitude),
                    longitude: segment.start.longitude
                        + segmentFraction * (segment.end.longitude - segment.start.longitude),
                    distanceAlongRouteMeters: targetDistance,
                    bearingDegrees: routeBearingDegrees(segment.start, segment.end)
                });
            }

            return { routeDistanceMeters, routeSamples };
        }

        function pointToSegmentDistanceMeters(point, start, end) {
            const latitudeScale = 111320;
            const longitudeScale = latitudeScale * Math.cos(point.latitude * Math.PI / 180);
            const startX = (start[0] - point.longitude) * longitudeScale;
            const startY = (start[1] - point.latitude) * latitudeScale;
            const endX = (end[0] - point.longitude) * longitudeScale;
            const endY = (end[1] - point.latitude) * latitudeScale;
            const segmentX = endX - startX;
            const segmentY = endY - startY;
            const segmentLengthSquared = segmentX ** 2 + segmentY ** 2;
            const fraction = segmentLengthSquared === 0
                ? 0
                : Math.min(1, Math.max(0, -(startX * segmentX + startY * segmentY) / segmentLengthSquared));
            return Math.hypot(
                startX + fraction * segmentX,
                startY + fraction * segmentY
            );
        }

        function distanceToHpmsLineMeters(point, lineCoordinates) {
            if (!Array.isArray(lineCoordinates) || lineCoordinates.length < 2) {
                return Number.POSITIVE_INFINITY;
            }
            return lineCoordinates.slice(1).reduce(
                (smallestDistance, coordinate, index) => Math.min(
                    smallestDistance,
                    pointToSegmentDistanceMeters(point, lineCoordinates[index], coordinate)
                ),
                Number.POSITIVE_INFINITY
            );
        }

        function nearestHpmsMatch(routeSample, hpmsSegments) {
            const matches = hpmsSegments.map((segment) => ({
                iri: Number(segment.iri),
                iriDate: segment.iri_d,
                distanceMeters: distanceToHpmsLineMeters(
                    routeSample,
                    segment.line && segment.line.coordinates
                )
            })).filter((match) => Number.isFinite(match.iri));
            if (matches.length === 0) {
                return null;
            }
            const nearestMatch = matches.reduce((nearest, match) => (
                match.distanceMeters < nearest.distanceMeters ? match : nearest
            ));
            return nearestMatch.distanceMeters <= 250 ? nearestMatch : null;
        }

        async function fetchHpmsMatch(routeSample) {
            const searchParameters = new URLSearchParams({
                "$select": "iri,iri_d,sectionlength,surface_type,route_id,routename,line",
                "$where": "iri IS NOT NULL AND within_circle(line, "
                    + routeSample.latitude + ", " + routeSample.longitude + ", 250)",
                "$limit": "100"
            });
            const response = await fetch(hpmsApiUrl + "?" + searchParameters);
            if (!response.ok) {
                throw new Error("HPMS returned " + response.status + ".");
            }
            return nearestHpmsMatch(routeSample, await response.json());
        }

        function historicDateStrings(raceDate) {
            const dateParts = raceDate.split("-").map(Number);
            const targetYear = dateParts[0];
            const month = dateParts[1];
            const day = dateParts[2];
            return [1, 2, 3].map((yearOffset) => {
                const year = targetYear - yearOffset;
                const daysInMonth = new Date(Date.UTC(year, month, 0)).getUTCDate();
                const validDay = Math.min(day, daysInMonth);
                return String(year) + "-" + String(month).padStart(2, "0")
                    + "-" + String(validDay).padStart(2, "0");
            });
        }

        function weatherRouteSamples() {
            const allRouteSamples = routeSamplesFromPoints(routePoints).routeSamples;
            const weatherSampleCount = Math.min(16, allRouteSamples.length);
            if (weatherSampleCount === 1) {
                return [allRouteSamples[0]];
            }
            return Array.from({ length: weatherSampleCount }, (_, index) => (
                allRouteSamples[Math.round(
                    index * (allRouteSamples.length - 1) / (weatherSampleCount - 1)
                )]
            ));
        }

        function airDensityKgM3(temperatureC, humidityPercent, pressureHpa) {
            if (![temperatureC, humidityPercent, pressureHpa].every(Number.isFinite)) {
                return null;
            }
            const temperatureKelvin = temperatureC + 273.15;
            const saturationPressureHpa = 6.112 * Math.exp(
                17.67 * temperatureC / (temperatureC + 243.5)
            );
            const vaporPressurePa = humidityPercent / 100 * saturationPressureHpa * 100;
            const dryAirPressurePa = pressureHpa * 100 - vaporPressurePa;
            return dryAirPressurePa / (287.05 * temperatureKelvin)
                + vaporPressurePa / (461.495 * temperatureKelvin);
        }

        function windComponentsMps(windSpeedKmh, windDirectionDegrees) {
            if (![windSpeedKmh, windDirectionDegrees].every(Number.isFinite)) {
                return null;
            }
            const windToRadians = (windDirectionDegrees + 180) * Math.PI / 180;
            const windSpeedMps = windSpeedKmh / 3.6;
            return {
                eastMps: windSpeedMps * Math.sin(windToRadians),
                northMps: windSpeedMps * Math.cos(windToRadians)
            };
        }

        function average(values, fallbackValue) {
            const validValues = values.filter(Number.isFinite);
            return validValues.length === 0
                ? fallbackValue
                : validValues.reduce((total, value) => total + value, 0) / validValues.length;
        }

        async function fetchHistoricWeather(date, routeSamples) {
            const weatherParameters = new URLSearchParams({
                latitude: routeSamples.map((sample) => sample.latitude).join(","),
                longitude: routeSamples.map((sample) => sample.longitude).join(","),
                start_date: date,
                end_date: date,
                hourly: "temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,wind_direction_10m",
                timezone: "auto"
            });
            const response = await fetch(weatherApiUrl + "?" + weatherParameters);
            if (!response.ok) {
                throw new Error("Historic weather returned " + response.status + ".");
            }
            const weatherData = await response.json();
            return Array.isArray(weatherData) ? weatherData : [weatherData];
        }

        function weatherProfileFromResponses(
            routeSamples,
            historicWeatherResponses,
            routeDistanceMiles,
            startTimeMinutes,
            historicDates
        ) {
            const weatherSamples = routeSamples.map((routeSample, sampleIndex) => ({
                distance_miles: routeSample.distanceAlongRouteMeters / metersPerMile,
                bearing_degrees: routeSample.bearingDegrees,
                air_density_kg_m3: Array.from({ length: 24 }, (_, hourIndex) => (
                    average(
                        historicWeatherResponses.map((yearResponse) => {
                            const hourly = yearResponse[sampleIndex].hourly;
                            return airDensityKgM3(
                                Number(hourly.temperature_2m[hourIndex]),
                                Number(hourly.relative_humidity_2m[hourIndex]),
                                Number(hourly.surface_pressure[hourIndex])
                            );
                        }),
                        1.225
                    )
                )),
                wind_east_mps: Array.from({ length: 24 }, (_, hourIndex) => (
                    average(
                        historicWeatherResponses.map((yearResponse) => {
                            const hourly = yearResponse[sampleIndex].hourly;
                            const wind = windComponentsMps(
                                Number(hourly.wind_speed_10m[hourIndex]),
                                Number(hourly.wind_direction_10m[hourIndex])
                            );
                            return wind ? wind.eastMps : null;
                        }),
                        0
                    )
                )),
                wind_north_mps: Array.from({ length: 24 }, (_, hourIndex) => (
                    average(
                        historicWeatherResponses.map((yearResponse) => {
                            const hourly = yearResponse[sampleIndex].hourly;
                            const wind = windComponentsMps(
                                Number(hourly.wind_speed_10m[hourIndex]),
                                Number(hourly.wind_direction_10m[hourIndex])
                            );
                            return wind ? wind.northMps : null;
                        }),
                        0
                    )
                ))
            }));
            return {
                historic_dates: historicDates,
                route_distance_miles: routeDistanceMiles,
                start_time_minutes: startTimeMinutes,
                samples: weatherSamples
            };
        }

        async function mapWithConcurrency(values, concurrency, mapValue) {
            const mappedValues = new Array(values.length);
            let nextIndex = 0;
            const workers = Array.from(
                { length: Math.min(concurrency, values.length) },
                async () => {
                    while (nextIndex < values.length) {
                        const currentIndex = nextIndex;
                        nextIndex += 1;
                        mappedValues[currentIndex] = await mapValue(
                            values[currentIndex],
                            currentIndex
                        );
                    }
                }
            );
            await Promise.all(workers);
            return mappedValues;
        }

        function pavementIriInchesPerMile() {
            return {
                "New pavement": 40,
                "Worn pavement": 80,
                "Poor pavement": 160,
                "Firm gravel": 250,
                "Rough gravel": 250,
                Cobbles: 250
            }[document.getElementById("surface").value];
        }

        async function analyzeRouteRoughness() {
            if (routePoints.length < 2) {
                return;
            }
            const routeDetails = routeSamplesFromPoints(routePoints);
            const routeDistanceMeters = routeDetails.routeDistanceMeters;
            const routeSamples = routeDetails.routeSamples;
            analyzeRouteButton.disabled = true;
            calculateButton.disabled = true;
            let completedSamples = 0;
            updateRouteStatus(
                "Matching " + routeSamples.length + " route samples to federal road data…",
                "loading"
            );
            const sampleMatches = await mapWithConcurrency(
                routeSamples,
                6,
                async (routeSample) => {
                    try {
                        return await fetchHpmsMatch(routeSample);
                    } catch (error) {
                        return null;
                    } finally {
                        completedSamples += 1;
                        updateRouteStatus(
                            "Matching route samples: " + completedSamples
                                + " of " + routeSamples.length + "…",
                            "loading"
                        );
                    }
                }
            );
            const matchedSamples = sampleMatches.filter(
                (sampleMatch) => sampleMatch !== null
            );
            const coverage = matchedSamples.length / routeSamples.length;
            if (matchedSamples.length === 0) {
                routeProfile = null;
                updateRouteStatus(
                    "No nearby HPMS roughness measurements were available. The selected pavement condition is still in use.",
                    "error"
                );
                analyzeRouteButton.disabled = false;
                calculateButton.disabled = false;
                return;
            }
            const measuredIri = matchedSamples.reduce(
                (total, sampleMatch) => total + sampleMatch.iri,
                0
            ) / matchedSamples.length;
            const effectiveIri = measuredIri * coverage
                + pavementIriInchesPerMile() * (1 - coverage);
            const iriYears = [...new Set(
                matchedSamples.map((sampleMatch) => (
                    sampleMatch.iriDate || ""
                ).slice(0, 4))
            )].filter(Boolean).join(", ");
            routeProfile = {
                routeIriInchesPerMile: effectiveIri,
                coverage,
                routeDistanceMeters,
                matchedSamples: matchedSamples.length,
                sampleCount: routeSamples.length
            };
            updateRouteStatus(
                (routeDistanceMeters / metersPerMile).toFixed(1) + " mi route · "
                    + matchedSamples.length + " of " + routeSamples.length
                    + " samples matched · " + (coverage * 100).toFixed(0)
                    + "% HPMS coverage · " + effectiveIri.toFixed(0)
                    + " in/mi modeled IRI"
                    + (iriYears ? " · measured " + iriYears : "") + ".",
                "ready"
            );
            analyzeRouteButton.disabled = false;
            await calculate(false);
        }

        async function analyzeHistoricWeather() {
            if (routePoints.length < 2 || !raceDateInput.value) {
                return;
            }
            const routeDetails = routeSamplesFromPoints(routePoints);
            const routeSamples = weatherRouteSamples();
            const historicDates = historicDateStrings(raceDateInput.value);
            const startTimeParts = raceStartTimeInput.value.split(":").map(Number);
            const startTimeMinutes = startTimeParts[0] * 60 + startTimeParts[1];
            analyzeWeatherButton.disabled = true;
            calculateButton.disabled = true;
            updateWeatherStatus(
                "Loading " + historicDates.join(", ") + " weather at "
                    + routeSamples.length + " course positions…",
                "loading"
            );
            try {
                const historicWeatherResponses = await Promise.all(
                    historicDates.map((date) => fetchHistoricWeather(date, routeSamples))
                );
                weatherProfile = weatherProfileFromResponses(
                    routeSamples,
                    historicWeatherResponses,
                    routeDetails.routeDistanceMeters / metersPerMile,
                    startTimeMinutes,
                    historicDates
                );
                await calculate(false);
            } catch (error) {
                weatherProfile = null;
                updateWeatherStatus(error.message, "error");
            } finally {
                updateWeatherButton();
                calculateButton.disabled = false;
            }
        }

        routeFileInput.addEventListener("change", async () => {
            routeProfile = null;
            weatherProfile = null;
            const routeFile = routeFileInput.files[0];
            if (!routeFile) {
                routePoints = [];
                analyzeRouteButton.disabled = true;
                updateWeatherButton();
                updateRouteStatus(
                    "Upload a GPX to match roughly one-mile route samples to FHWA HPMS road-roughness data."
                );
                return;
            }
            try {
                routePoints = parseGpxRoute(await routeFile.text());
                const routeDetails = routeSamplesFromPoints(routePoints);
                analyzeRouteButton.disabled = !pythonRuntime;
                updateWeatherButton();
                updateRouteStatus(
                    routeFile.name + " is ready: "
                        + (routeDetails.routeDistanceMeters / metersPerMile).toFixed(1)
                        + " mi and " + routeDetails.routeSamples.length
                        + " roughness samples.",
                    "ready"
                );
            } catch (error) {
                routePoints = [];
                analyzeRouteButton.disabled = true;
                updateWeatherButton();
                updateRouteStatus(error.message, "error");
            }
        });

        raceDateInput.addEventListener("change", () => {
            weatherProfile = null;
            updateWeatherButton();
            updateWeatherStatus(
                "Use the same calendar date from the three prior years after choosing a GPX."
            );
        });

        raceStartTimeInput.addEventListener("change", () => {
            if (weatherProfile) {
                weatherProfile = null;
                updateWeatherStatus(
                    "Start time changed. Reload historic weather to update each course point’s arrival hour."
                );
            }
        });

        function numericValue(elementId) {
            return Number(document.getElementById(elementId).value);
        }

        function calculatorInputs() {
            return {
                system_weight_kg: numericValue("system-weight") * 0.45359237,
                rider_power_watts: numericValue("rider-power"),
                wheel_bead_diameter_mm: numericValue("wheel-size"),
                bike_name: document.getElementById("bike").value,
                rider_height_cm: numericValue("rider-height-feet") * 30.48
                    + numericValue("rider-height-inches") * 2.54,
                shoulder_width_cm: numericValue("shoulder-width"),
                cockpit_width_cm: numericValue("cockpit-width"),
                kit_type: document.getElementById("kit-type").value,
                surface_name: document.getElementById("surface").value,
                route_iri_inches_per_mile: routeProfile
                    ? routeProfile.routeIriInchesPerMile
                    : null,
                weather_profile: weatherProfile,
                front_internal_width_mm: numericValue("front-internal"),
                front_external_width_mm: numericValue("front-external"),
                front_rim_depth_mm: numericValue("front-depth"),
                rear_internal_width_mm: numericValue("rear-internal"),
                rear_external_width_mm: numericValue("rear-external"),
                rear_rim_depth_mm: numericValue("rear-depth")
            };
        }

        function fixedNumber(value) {
            return Number(value).toFixed(1);
        }

        function renderWork(winner, inputs) {
            const roadPowerWatts = winner.rider_power_watts * winner.drivetrain_efficiency;
            const riderBikeAeroWatts = roadPowerWatts - winner.rolling_watts - winner.surface_watts - winner.aero_watts;
            const frontRoadLossWatts = winner.front_rolling_watts + winner.front_surface_watts;
            const rearRoadLossWatts = winner.rear_rolling_watts + winner.rear_surface_watts;
            const speedMetersPerSecond = winner.predicted_speed_kmh / 3.6;
            const signedWatts = (watts) => (watts >= 0 ? "+" : "") + fixedNumber(watts) + " W";
            const routeIri = inputs.route_iri_inches_per_mile === null ? "selected " + inputs.surface_name : fixedNumber(inputs.route_iri_inches_per_mile) + " in/mi";
            const routeDescription = inputs.route_iri_inches_per_mile === null ? "the selected " + inputs.surface_name.toLowerCase() + " surface" : "the GPX-matched HPMS roughness of " + routeIri;
            const weatherDescription = inputs.weather_profile ? fixedNumber(winner.air_density_kg_m3) + " kg/m³ and " + fixedNumber(winner.mean_apparent_yaw_degrees) + "° yaw" : "standard 1.225 kg/m³ air and no route-weather yaw";
            workPanel.innerHTML = `
                <p class="section-label">Calculation for the current winner</p>
                <h2>${winner.front_tire} front / ${winner.rear_tire} rear</h2>
                <section class="work-recommendation">
                    <h3>Your recommendation, in numbers</h3>
                    <p>Based on your inputs, the reconstructed SILCA-derived pressure model recommends <strong>${fixedNumber(winner.front_pressure_psi)} psi</strong> in the front and <strong>${fixedNumber(winner.rear_pressure_psi)} psi</strong> in the rear. The predicted inflated sizes are <strong>${fixedNumber(winner.front_width_mm)} mm</strong> front and <strong>${fixedNumber(winner.rear_width_mm)} mm</strong> rear.</p>
                    <p>In the calculator’s BRR-style steel-drum reference—one tire, 42.5 kg load and 29.0 km/h (18.0 mph)—the pressure-adjusted estimates are <strong>${fixedNumber(winner.front_brr_reference_watts)} W</strong> for the front and <strong>${fixedNumber(winner.rear_brr_reference_watts)} W</strong> for the rear. This is a standardized comparison reference, not a prediction of road loss.</p>
                    <p>At the modeled speed and with ${routeDescription}, the front contributes <strong>${fixedNumber(winner.front_rolling_watts)} W</strong> casing rolling loss plus <strong>${fixedNumber(winner.front_surface_watts)} W</strong> surface loss. The rear contributes <strong>${fixedNumber(winner.rear_rolling_watts)} W</strong> casing rolling loss plus <strong>${fixedNumber(winner.rear_surface_watts)} W</strong> surface loss. That is <strong>${fixedNumber(frontRoadLossWatts + rearRoadLossWatts)} W</strong> of road-related tire loss combined.</p>
                    <p>The tire-rim aero adjustment is <strong>${signedWatts(winner.front_aero_watts)}</strong> at the front and <strong>${signedWatts(winner.rear_aero_watts)}</strong> at the rear relative to the GP5000 S TR 28 baseline. A negative value is an aero credit. The model deliberately weights the equivalent front effect <strong>5×</strong> the rear effect (1.0 vs 0.2 exposure): that is a transparent model assumption, not a universal wind-tunnel finding. Parcours and Cyclingnews supply the tire/wheel aero evidence; historic weather supplies ${weatherDescription}. The current scalar tire-aero adjustment does not yet change with the calculated yaw.</p>
                    <p>For rider-bike aero, the calculator estimates <strong>${winner.rider_bike_cda_m2.toFixed(3)} m² CdA</strong> from selected riding position, height, shoulder width, cockpit width and kit type. This is a transparent directional estimate, not a personal wind-tunnel or field test. The drivetrain factor is currently a <strong>${fixedNumber(winner.drivetrain_efficiency * 100)}%</strong> model assumption, not a TK-sourced measurement.</p>
                    <p>At <strong>${fixedNumber(winner.rider_power_watts)} W average pedal power for the whole course</strong>—not Normalized Power—the solved balance is:</p>
                    <code>${fixedNumber(roadPowerWatts)} W = 0.5 × ${winner.air_density_kg_m3.toFixed(3)} kg/m³ × ${winner.rider_bike_cda_m2.toFixed(2)} m² × (${speedMetersPerSecond.toFixed(2)} m/s)³ + ${fixedNumber(winner.front_rolling_watts)} W + ${fixedNumber(winner.rear_rolling_watts)} W + ${fixedNumber(winner.front_surface_watts)} W + ${fixedNumber(winner.rear_surface_watts)} W + ${signedWatts(winner.front_aero_watts)} + ${signedWatts(winner.rear_aero_watts)}</code>
                    <p>Therefore this tire pair’s modeled steady speed is <strong>${fixedNumber(winner.predicted_speed_mph)} mph</strong>.</p>
                </section>
                <section class="work-explainer">
                    <h3>Why this math, and the order it happens</h3>
                    <p>There is no fixed speed to start from: speed changes rolling loss, surface loss and aerodynamic drag. The model therefore evaluates a tire pair at trial speeds and finds the speed that your power can sustain.</p>
                    <ol class="work-steps">
                        <li><strong>Build the setup.</strong> Convert each quoted tire size into its predicted mounted width on your internal rim width. Width determines air volume, contact patch and how the tire meets the rim.</li>
                        <li><strong>Set pressure.</strong> Use system mass, front/rear load split, mounted width, bead diameter and road roughness. This gives a pressure appropriate to the tire rather than assuming one number works for every setup.</li>
                        <li><strong>Set conditions.</strong> GPX roughness changes surface impedance. Historical weather, when loaded, changes air density and calculates the apparent-wind yaw at each course point.</li>
                        <li><strong>Calculate losses at a trial speed.</strong> Add rolling resistance, surface impedance, tire-rim aero and rider-bike aero. Wider tires can reduce surface loss while adding frontal-area or rim-transition drag.</li>
                        <li><strong>Match power to resistance.</strong> Convert pedal power to road power using 97% drivetrain efficiency. Increase or decrease trial speed until road power equals all four losses.</li>
                        <li><strong>Repeat and rank.</strong> Run the same solve for every valid front/rear tire pair, then rank them by the highest sustainable steady speed. That is why the winner is a tire system, not just a tire.</li>
                    </ol>
                </section>
                <div class="work-grid">
                    <article class="work-card">
                        <h3>Inputs at the solution</h3>
                        <div class="work-values">
                            <span>Pedal power<strong>${fixedNumber(winner.rider_power_watts)} W</strong></span><span>Road power<strong>${fixedNumber(roadPowerWatts)} W</strong></span>
                            <span>Speed<strong>${fixedNumber(winner.predicted_speed_mph)} mph</strong></span><span>CdA<strong>${winner.rider_bike_cda_m2.toFixed(2)} m²</strong></span>
                            <span>Front mounted<strong>${fixedNumber(winner.front_width_mm)} mm / ${fixedNumber(winner.front_pressure_psi)} psi</strong></span><span>Rear mounted<strong>${fixedNumber(winner.rear_width_mm)} mm / ${fixedNumber(winner.rear_pressure_psi)} psi</strong></span>
                            <span>Road roughness<strong>${routeIri}</strong></span><span>Weather<strong>${weatherDescription}</strong></span>
                        </div>
                    </article>
                    <article class="work-card">
                        <h3>Power balance</h3>
                        <div class="work-values">
                            <span>Rider-bike aero<strong>${fixedNumber(riderBikeAeroWatts)} W</strong></span><span>Rolling resistance<strong>${fixedNumber(winner.rolling_watts)} W</strong></span>
                            <span>Surface impedance<strong>${fixedNumber(winner.surface_watts)} W</strong></span><span>Tire-rim aero<strong>${fixedNumber(winner.aero_watts)} W</strong></span>
                        </div>
                        <p>The solver finds the speed where delivered road power equals the four terms shown here.</p>
                        <code>0.97 × pedal power = 0.5 × ρ × CdA × v³ + rolling + surface + tire aero</code>
                    </article>
                    <article class="work-card">
                        <h3>Mounted width and pressure</h3>
                        <code>width = measured width + slope × (internal rim width − reference internal width)</code>
                        <p>SL-R 28 follows the supplied Pirelli 19c/21c/23c WAM chart by linear interpolation.</p>
                        <code>pressure = clamp(contact-patch pressure × speed coefficient × axle coefficient, 28, 120)</code>
                        <p>Contact-patch pressure is the reconstructed SILCA-derived relationship in the linked build notebook, using system mass, bead diameter, mounted width and surface coefficient.</p>
                    </article>
                    <article class="work-card">
                        <h3>Loss terms</h3>
                        <code>Crrref = BRR reference watts ÷ (42.5 kg × 9.80665 × 29/3.6 m/s)</code>
                        <code>rolling = Crrref × (reference psi ÷ actual psi)^0.12 × load × g × v</code>
                        <code>surface = roughness × load × (v/40)^2.2 × (30/width)^3</code>
                        <code>tire aero = rim/tire baseline × (v/40)^3 × (ρ/1.225) × axle factor</code>
                        <p>TT TR and S TR share the conventional-tire yaw treatment. AERO 111 and SL-R have published high-yaw behavior, but no universal curve is transferred here across every rim.</p>
                    </article>
                </div>`;
        }

        showWorkButton.addEventListener("click", () => {
            const shouldShowWork = workPanel.hidden;
            workPanel.hidden = !shouldShowWork;
            showWorkButton.setAttribute("aria-expanded", String(shouldShowWork));
            showWorkButton.textContent = shouldShowWork ? "Hide your work" : "Show your work";
        });

        function renderResults(results) {
            const winner = results[0];
            latestWinner = winner;
            latestInputs = calculatorInputs();
            renderWork(latestWinner, latestInputs);
            workPanel.hidden = true;
            showWorkButton.disabled = false;
            showWorkButton.setAttribute("aria-expanded", "false");
            showWorkButton.textContent = "Show your work";
            document.getElementById("winner-output").innerHTML = `
                <section class="winner-card" data-testid="winner-card">
                    <div class="section-label">Fastest modeled system</div>
                    <div class="winner-grid">
                        <div><small>FRONT</small><h3>${winner.front_tire}</h3><p>${fixedNumber(winner.front_pressure_psi)} psi · ${fixedNumber(winner.front_width_mm)} mm mounted</p></div>
                        <div><small>REAR</small><h3>${winner.rear_tire}</h3><p>${fixedNumber(winner.rear_pressure_psi)} psi · ${fixedNumber(winner.rear_width_mm)} mm mounted</p></div>
                    </div>
                    <div class="winner-total"><strong data-testid="predicted-speed">${fixedNumber(winner.predicted_speed_mph)} mph</strong><span> modeled steady speed at ${fixedNumber(winner.rider_power_watts)} W average pedal power</span></div>
                </section>`;

            const headings = ["Rank", "Front", "Rear", "Speed mph", "Gap mph", "Front psi", "Rear psi", "Front mm", "Rear mm", "Tire loss W", "Confidence"];
            const rows = results.map((result) => [
                result.rank,
                result.front_tire,
                result.rear_tire,
                fixedNumber(result.predicted_speed_mph),
                fixedNumber(result.gap_speed_mph),
                fixedNumber(result.front_pressure_psi),
                fixedNumber(result.rear_pressure_psi),
                fixedNumber(result.front_width_mm),
                fixedNumber(result.rear_width_mm),
                fixedNumber(result.tire_loss_watts),
                result.confidence
            ]);
            document.getElementById("ranking-table").innerHTML = `
                <thead><tr>${headings.map((heading) => `<th>${heading}</th>`).join("")}</tr></thead>
                <tbody>${rows.map((row) => `<tr>${row.map((value) => `<td>${value}</td>`).join("")}</tr>`).join("")}</tbody>`;
            if (weatherProfile) {
                updateWeatherStatus(
                    "Historic weather from " + weatherProfile.historic_dates.join(", ")
                        + " at " + weatherProfile.samples.length + " course positions · "
                        + winner.air_density_kg_m3.toFixed(3) + " kg/m³ air density · "
                        + winner.mean_apparent_yaw_degrees.toFixed(1)
                        + "° mean apparent-wind yaw at "
                        + winner.predicted_speed_mph.toFixed(1) + " mph.",
                    "ready"
                );
            }
        }

        async function calculate(showResults = true) {
            calculateButton.disabled = true;
            calculateButton.textContent = "Calculating…";
            try {
                pythonRuntime.globals.set(
                    "calculator_input_json",
                    JSON.stringify(calculatorInputs())
                );
                const resultJson = pythonRuntime.runPython(
                    "calculate_json(calculator_input_json)"
                );
                renderResults(JSON.parse(resultJson));
                resultsTab.disabled = false;
                if (showResults) {
                    showWizardStep(4);
                }
                runtimeStatus.textContent = "Python model ready";
                runtimeStatus.dataset.state = "ready";
                document.body.dataset.ready = "true";
            } catch (error) {
                runtimeStatus.textContent = `Calculation failed: ${error.message}`;
                runtimeStatus.dataset.state = "error";
                document.body.dataset.ready = "error";
                throw error;
            } finally {
                calculateButton.disabled = false;
                calculateButton.textContent = "Calculate and view results";
            }
        }

        async function initializeCalculator() {
            try {
                pythonRuntime = await loadPyodide();
                await pythonRuntime.runPythonAsync(pythonModelSource);
                calculateButton.addEventListener("click", calculate);
                analyzeRouteButton.addEventListener("click", analyzeRouteRoughness);
                analyzeWeatherButton.addEventListener("click", analyzeHistoricWeather);
                analyzeRouteButton.disabled = routePoints.length < 2;
                updateWeatherButton();
                calculateButton.disabled = false;
                calculateButton.textContent = "Calculate and view results";
                runtimeStatus.textContent = "Python model ready";
                runtimeStatus.dataset.state = "ready";
                document.body.dataset.ready = "true";
            } catch (error) {
                runtimeStatus.textContent = `Model failed to load: ${error.message}`;
                runtimeStatus.dataset.state = "error";
                document.body.dataset.ready = "error";
                console.error(error);
            }
        }

        initializeCalculator();
    </script>
</body>
</html>
'''
standalone_html = standalone_html.replace(
    "__PYTHON_MODEL_SOURCE__",
    json.dumps(browser_model_source),
)

with open("../docs/index.html", "w", encoding="utf-8") as output_file:
    output_file.write(standalone_html)

Path(
    "../docs/assets/fixed/gradio-5.45.0-cp312-none-any.whl"
).unlink(missing_ok=True)

## Checks

In [14]:
assert len(tire_specs) == 11
assert default_ranking.shape[0] == 99
assert default_ranking["predicted_speed_kmh"].is_monotonic_decreasing
assert default_ranking["predicted_speed_mph"].between(5, 50).all()
assert Path("../docs/index.html").exists()
assert not Path(
    "../docs/assets/fixed/gradio-5.45.0-cp312-none-any.whl"
).exists()

with open("../docs/index.html", encoding="utf-8") as output_file:
    generated_html = output_file.read()

assert "pyodide/v0.27.7/full/pyodide.js" in generated_html
assert "data-testid=\"winner-card\"" in generated_html
assert "@gradio/lite" not in generated_html
assert "huggingface" not in generated_html.lower()
assert "Continental GP5000 S TR 35" in generated_html
assert "Pirelli P Zero Race TLR SL-R 30" in generated_html
assert "calculate_json(calculator_input_json)" in generated_html
assert "29.773" in generated_html
assert "31.414" in generated_html
assert "not a 105% rule" in generated_html
assert "Average pedal power for the course (W)" in generated_html
assert "1. Tire pressure" in generated_html
assert "2. Rider and bike aero" in generated_html
assert "3. Course and weather" in generated_html
assert "4. Results" in generated_html
assert "Do you want to upload a GPX?" in generated_html
assert "What time will you start biking?" in generated_html
assert "id=\"results-section\" class=\"wizard-step\" data-wizard-step=\"4\"" in generated_html
assert "rider-height-feet" in generated_html
assert "rider-height-inches" in generated_html
assert "Shoulder width (cm)" in generated_html
assert "Handlebar / arm-pad width (cm)" in generated_html
assert "kit-type" in generated_html
assert "wheel-size" in generated_html
assert "route-file" in generated_html
assert "HPMS road-roughness data" in generated_html
assert "route_iri_inches_per_mile" in generated_html
assert "race-date" in generated_html
assert "archive-api.open-meteo.com" in generated_html
assert "weather_profile" in generated_html
assert "ranked_tires[:10]" not in generated_html

{
    "tire_count": len(tire_specs),
    "pairing_count": default_ranking.shape[0],
    "default_winner": default_ranking.iloc[0][
        ["front_tire", "rear_tire", "predicted_speed_mph"]
    ].to_dict(),
    "static_app_bytes": Path("../docs/index.html").stat().st_size,
}

{'tire_count': 11,
 'pairing_count': 99,
 'default_winner': {'front_tire': 'SL-R 28',
  'rear_tire': 'SL-R 28',
  'predicted_speed_mph': 23.971748699786136},
 'static_app_bytes': 99941}

## Next Steps

The notebook and the generated Pages artifact are ready to publish. Medium
confidence pairings should be treated as a short list for real-world testing
because public wind-tunnel coverage is incomplete.